## 🔧 第一步：导入依赖

# 🦴 Medical-Bone-Age 骨龄预测训练系统

三阶段递进式骨龄预测训练框架

## 📋 训练流程
1. **Stage1**: 源域特征学习 (Arthritis关节分类)
2. **Stage2**: 域自适应对齐 (GRL + MMD)
3. **Stage3**: 公式法骨龄精调 (RSNA数据集 + DPV3关节分割)

In [1]:
import os
import sys
import json
import math
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score, mean_absolute_error
import matplotlib.pyplot as plt

print(f"PyTorch版本: {torch.__version__}")
print(f"设备: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

PyTorch版本: 2.8.0+cu126
设备: CUDA


## ⚙️ 第二步：导入自定义模块

In [2]:
'''
# 导入配置
from config import Config, PositionalEncoding2D

# 导入模型
from models import CNNTransformerBoneAge, GradientReversalLayer

# 导入损失函数
from losses import MMDLoss, calc_bone_age_from_score, compute_formula_weights

# 导入数据集
from datasets import ArthrosisDataset, RSNADataset

# 导入数据工具
from data_utils import split_arthrosis_dataset, prepare_rsna_dataset

# 导入训练函数
from training import (
    get_grl_lambda,
    stage1_source_feature_learning,
    stage2_domain_adaptation,
    stage3_formula_bone_age
)

# 导入评估函数
from evaluation import (
    evaluate_joint_classification,
    evaluate_domain_loss,
    evaluate_bone_age_mae
)

print("✅ 所有模块导入成功！")
'''

'\n# 导入配置\nfrom config import Config, PositionalEncoding2D\n\n# 导入模型\nfrom models import CNNTransformerBoneAge, GradientReversalLayer\n\n# 导入损失函数\nfrom losses import MMDLoss, calc_bone_age_from_score, compute_formula_weights\n\n# 导入数据集\nfrom datasets import ArthrosisDataset, RSNADataset\n\n# 导入数据工具\nfrom data_utils import split_arthrosis_dataset, prepare_rsna_dataset\n\n# 导入训练函数\nfrom training import (\n    get_grl_lambda,\n    stage1_source_feature_learning,\n    stage2_domain_adaptation,\n    stage3_formula_bone_age\n)\n\n# 导入评估函数\nfrom evaluation import (\n    evaluate_joint_classification,\n    evaluate_domain_loss,\n    evaluate_bone_age_mae\n)\n\nprint("✅ 所有模块导入成功！")\n'

In [3]:
"""
配置模块

包含:
- Config: 全局配置类
- PositionalEncoding2D: 2D余弦位置编码
"""

import os
import math

import torch
import torch.nn as nn


def _directory_has_subdirs(path: str) -> bool:
    """目录存在且至少包含一个子目录。"""
    if not os.path.isdir(path):
        return False

    try:
        return any(os.path.isdir(os.path.join(path, name)) for name in os.listdir(path))
    except OSError:
        return False


def _resolve_existing_path(preferred: str, candidates: list[str], validator) -> str:
    """优先使用首选路径，否则回退到第一个可用候选路径。"""
    if preferred and validator(preferred):
        return preferred

    for candidate in candidates:
        if candidate and validator(candidate):
            return candidate

    return preferred


class Config:
    """全局配置类"""

  
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    SOURCE_DATA_ROOT = r"/kaggle/input/private-dataset/arthrosis"
    TARGET_DATA_ROOT =r"/kaggle/input/datasets/lihaohao111/jointsadaptation/joints_output"
    RSNA_CSV_PATH = r"/kaggle/input/rsna-bone-age/boneage-training-dataset.csv"
    RSNA_IMG_DIR = r"/kaggle/input/datasets/lihaohao111/jointsadaptation/joints_output"
    YOLO_JOINT_MODEL_PATH = r"/kaggle/input/models/lihaohao111/bestyolo/pytorch/default/1/best (1).pt"
    
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    IMG_SIZE = 256
    D_MODEL = 512
    NHEAD = 8
    NUM_LAYERS = 6
    DROPOUT = 0.1
    
    BATCH_SIZE = 32
    EPOCHS_STAGE1 = 10
    EPOCHS_STAGE2 = 30
    EPOCHS_STAGE3 = 10
    LEARNING_RATE_STAGE1 = 1e-4
    LEARNING_RATE_STAGE2 = 5e-5
    LEARNING_RATE_STAGE3 = 1e-5
    
    RSNA_SAMPLES = 1000
    
    PHYSICAL_JOINT_NAMES = ['Radius', 'Ulna', 'MCPFirst', 'MCP', 'PIP', 'DIP', 'MIP', 'DIPFirst', 'PIPFirst']
    
    PHYSICAL_TO_MEDICAL_MAPPING = {
        'Radius': ['Radius'],
        'Ulna': ['Ulna'],
        'MCPFirst': ['MCPFirst'],
        'MCP': ['MCPThird', 'MCPFifth'],
        'PIP': ['PIPThird', 'PIPFifth'],
        'DIP': ['DIPThird', 'DIPFifth'],
        'MIP': ['MIPThird', 'MIPFifth'],
        'DIPFirst': ['DIPFirst'],
        'PIPFirst': ['PIPFirst']
    }
    
    MEDICAL_JOINT_NAMES = ['Radius', 'Ulna', 'MCPFirst', 'MCPThird', 'MCPFifth', 
                          'PIPFirst', 'PIPThird', 'PIPFifth', 
                          'MIPThird', 'MIPFifth',
                          'DIPFirst', 'DIPThird', 'DIPFifth']
    
    MEDICAL_JOINT_NUM_CLASSES = {
        'Radius': 15, 'Ulna': 13, 'MCPFirst': 12, 'MCPThird': 11, 'MCPFifth': 11,
        'PIPFirst': 13, 'PIPThird': 13, 'PIPFifth': 13,
        'MIPThird': 13, 'MIPFifth': 13,
        'DIPFirst': 12, 'DIPThird': 12, 'DIPFifth': 12
    }
    
    SCORE_TABLE = {
        'female': {
            'Radius': [0, 10, 15, 22, 25, 40, 59, 91, 125, 138, 178, 192, 199, 203, 210],
            'Ulna': [0, 27, 31, 36, 50, 73, 95, 120, 157, 168, 176, 182, 189],
            'MCPFirst': [0, 5, 7, 10, 16, 23, 28, 34, 41, 47, 53, 66],
            'MCPThird': [0, 3, 5, 6, 9, 14, 21, 32, 40, 47, 51],
            'MCPFifth': [0, 4, 5, 7, 10, 15, 22, 33, 43, 47, 51],
            'PIPFirst': [0, 6, 7, 8, 11, 17, 26, 32, 38, 45, 53, 60, 67],
            'PIPThird': [0, 3, 5, 7, 9, 15, 20, 25, 29, 35, 41, 46, 51],
            'PIPFifth': [0, 4, 5, 7, 11, 18, 21, 25, 29, 34, 40, 45, 50],
            'MIPThird': [0, 4, 5, 7, 10, 16, 21, 25, 29, 35, 43, 46, 51],
            'MIPFifth': [0, 3, 5, 7, 12, 19, 23, 27, 32, 35, 39, 43, 49],
            'DIPFirst': [0, 5, 6, 8, 10, 20, 31, 38, 44, 45, 52, 67],
            'DIPThird': [0, 3, 5, 7, 10, 16, 24, 30, 33, 36, 39, 49],
            'DIPFifth': [0, 5, 6, 7, 11, 18, 25, 29, 33, 35, 39, 49]
        },
        'male': {
            'Radius': [0, 8, 11, 15, 18, 31, 46, 76, 118, 135, 171, 188, 197, 201, 209],
            'Ulna': [0, 25, 30, 35, 43, 61, 80, 116, 157, 168, 180, 187, 194],
            'MCPFirst': [0, 4, 5, 8, 16, 22, 26, 34, 39, 45, 52, 66],
            'MCPThird': [0, 3, 4, 5, 8, 13, 19, 30, 38, 44, 51],
            'MCPFifth': [0, 3, 4, 6, 9, 14, 19, 31, 41, 46, 50],
            'PIPFirst': [0, 4, 5, 7, 11, 17, 23, 29, 36, 44, 52, 59, 66],
            'PIPThird': [0, 3, 4, 5, 8, 14, 19, 23, 28, 34, 40, 45, 50],
            'PIPFifth': [0, 3, 4, 6, 10, 16, 19, 24, 28, 33, 40, 44, 50],
            'MIPThird': [0, 3, 4, 5, 9, 14, 18, 23, 28, 35, 42, 45, 50],
            'MIPFifth': [0, 3, 4, 6, 11, 17, 21, 26, 31, 36, 40, 43, 49],
            'DIPFirst': [0, 4, 5, 6, 9, 19, 28, 36, 43, 46, 51, 67],
            'DIPThird': [0, 3, 4, 5, 9, 15, 23, 29, 33, 37, 40, 49],
            'DIPFifth': [0, 3, 4, 6, 11, 17, 23, 29, 32, 36, 40, 49]
        }
    }
    
    @classmethod
    def print_config(cls):
        print("=" * 60)
        print("               配置参数")
        print("=" * 60)
        print(f"设备: {cls.DEVICE}")
        print(f"源域路径: {cls.SOURCE_DATA_ROOT}")
        print(f"目标域路径: {cls.TARGET_DATA_ROOT}")
        print(f"RSNA CSV: {cls.RSNA_CSV_PATH}")
        print(f"RSNA图像: {cls.RSNA_IMG_DIR}")
        print(f"YOLO权重: {cls.YOLO_JOINT_MODEL_PATH}")
        print(f"图像尺寸: {cls.IMG_SIZE}")
        print(f"D_MODEL: {cls.D_MODEL}")
        print(f"NHEAD: {cls.NHEAD}")
        print(f"NUM_LAYERS: {cls.NUM_LAYERS}")
        print(f"Batch Size: {cls.BATCH_SIZE}")
        print(f"Stage1 Epochs: {cls.EPOCHS_STAGE1}")
        print(f"Stage2 Epochs: {cls.EPOCHS_STAGE2}")
        print(f"Stage3 Epochs: {cls.EPOCHS_STAGE3}")
        print("=" * 60)


class PositionalEncoding2D(nn.Module):
    """2D余弦位置编码，编码方式: n*x + y (矩阵简单相加融合)"""
    
    def __init__(self, d_model: int, height: int = 8, width: int = 8):
        super().__init__()
        self.d_model = d_model
        self.height = height
        self.width = width
        
        pe = self._create_sinusoidal_pe(d_model, height, width)
        self.register_buffer('pe', pe)
    
    def _create_sinusoidal_pe(self, d_model: int, height: int, width: int):
        """创建2D正弦位置编码"""
        pe = torch.zeros(d_model, height, width)
        
        n = width
        position_x = torch.arange(0, height).unsqueeze(1)
        position_y = torch.arange(0, width).unsqueeze(0)
        position = n * position_x + position_y
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        
        pe[0::2, :, :] = torch.sin(position.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        pe[1::2, :, :] = torch.cos(position.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        
        return pe
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        
        if h == 1 and w == 1:
            return x
        
        if self.pe.shape[1] != h or self.pe.shape[2] != w:
            pe = self._create_sinusoidal_pe(self.d_model, h, w).to(x.device)
        else:
            pe = self.pe
        
        return x + pe.unsqueeze(0)


In [4]:
"""
模型定义模块

包含:
- GradientReversalFunction: 梯度反转层
- GradientReversalLayer: 梯度反转层包装
- CNNTransformerBoneAge: CNN-Transformer骨龄分类模型
"""

import torch
import torch.nn as nn
from typing import Dict
from torchvision import models



class GradientReversalFunction(torch.autograd.Function):
    """梯度反转层 (Gradient Reversal Layer)"""
    
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_: float):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    
    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambda_ * grad_output, None


class GradientReversalLayer(nn.Module):
    """梯度反转层包装"""
    
    def __init__(self, lambda_: float = 1.0):
        super().__init__()
        self.lambda_ = lambda_
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return GradientReversalFunction.apply(x, self.lambda_)


class CNNTransformerBoneAge(nn.Module):
    """
    CNN + Transformer 骨龄分类模型
    
    特征融合方式：简单矩阵相加 (Addition)
    """
    
    def __init__(self, num_classes_dict: Dict[str, int], 
                 d_model: int = 512, nhead: int = 8, 
                 num_layers: int = 6, dropout: float = 0.1,
                 backbone: str = 'resnet50', height: int = 8, width: int = 8):
        super().__init__()
        
        self.d_model = d_model
        self.num_classes_dict = num_classes_dict
        
        if backbone == 'resnet50':
            resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            self.cnn = nn.Sequential(*list(resnet.children())[:-1])
            feat_channels = 2048
            spatial_size = 7
        elif backbone == 'resnet18':
            resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.cnn = nn.Sequential(*list(resnet.children())[:-1])
            feat_channels = 512
            spatial_size = 7
        elif backbone == 'efficientnet_b0':
            effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.cnn = nn.Sequential(effnet.features, effnet.avgpool)
            feat_channels = 1280
            spatial_size = 8
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
        self.cnn.eval()
        self.projection = nn.Conv2d(feat_channels, d_model, kernel_size=1)
        self.feat_channels = feat_channels
        self.spatial_size = spatial_size
        
        self.pos_encoder = PositionalEncoding2D(d_model=d_model, height=spatial_size, width=spatial_size)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classification_heads = nn.ModuleDict({
            name: nn.Linear(d_model, num_classes) 
            for name, num_classes in num_classes_dict.items()
        })
        
        self.domain_classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )
        self.grl_lambda = 1.0
    
    def forward(self, x: torch.Tensor, 
                return_domain: bool = False, 
                domain_lambda: float = 1.0) -> Dict[str, torch.Tensor]:
        """前向传播"""
        cnn_feat = self.cnn(x)
        
        proj_feat = self.projection(cnn_feat)
        
        pos_feat = self.pos_encoder(proj_feat)
        
        transformer_input = pos_feat.flatten(2).permute(0, 2, 1)
        transformer_output = self.transformer(transformer_input)
        
        global_feat = transformer_output.mean(dim=1)
        
        joint_logits = {}
        for name, head in self.classification_heads.items():
            joint_logits[name] = head(global_feat)
        
        if return_domain:
            reversed_feat = GradientReversalFunction.apply(global_feat, domain_lambda)
            domain_logits = self.domain_classifier(reversed_feat)
            return joint_logits, domain_logits
        
        return joint_logits
    
    def get_global_features(self, x: torch.Tensor) -> torch.Tensor:
        """获取全局特征（用于MMD计算）"""
        with torch.no_grad():
            cnn_feat = self.cnn(x)
            proj_feat = self.projection(cnn_feat)
            pos_feat = self.pos_encoder(proj_feat)
            transformer_input = pos_feat.flatten(2).permute(0, 2, 1)
            transformer_output = self.transformer(transformer_input)
            global_feat = transformer_output.mean(dim=1)
        return global_feat


In [5]:
"""
损失函数模块

包含:
- MMDLoss: 最大均值差异损失
- calc_bone_age_from_score: RUS-CHN评分转骨龄
- compute_formula_weights: 计算公式权重
"""

import torch
import torch.nn as nn
import math
from typing import Dict


class MMDLoss(nn.Module):
    """最大均值差异损失 (Maximum Mean Discrepancy)"""
    
    def __init__(self, sigma: float = 1.0):
        super().__init__()
        self.sigma = sigma
    
    def gaussian_kernel(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        diff = x.unsqueeze(1) - y.unsqueeze(0)
        return torch.exp(-torch.sum(diff ** 2, dim=-1) / (2 * self.sigma ** 2))
    
    def forward(self, source: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        K_ss = self.gaussian_kernel(source, source).mean()
        K_tt = self.gaussian_kernel(target, target).mean()
        K_st = self.gaussian_kernel(source, target).mean()
        return torch.clamp(K_ss + K_tt - 2 * K_st, min=0)


def compute_formula_weights() -> Dict[str, float]:
    """计算公式法各关节权重"""
    weights = {}
    joint_ranges = {
        'Radius': 15, 'Ulna': 13, 'MCPFirst': 12, 'MCPThird': 11,
        'MCPFifth': 11, 'PIPFirst': 13, 'PIPThird': 13, 'PIPFifth': 13,
        'MIPThird': 13, 'MIPFifth': 13, 'DIPFirst': 12, 'DIPThird': 12,
        'DIPFifth': 12
    }
    total = sum(joint_ranges.values())
    for joint, r in joint_ranges.items():
        weights[joint] = r / total
    return weights


def calc_bone_age_from_score(score: float, gender: str) -> float:
    """
    根据RUS-CHN评分计算骨龄（完整10次多项式）
    
    Args:
        score: RUS-CHN总分
        gender: 'male' 或 'female'
    Returns:
        骨龄（年）
    """
    if gender == 'male':
        boneAge = (
            2.01790023656577 
            + (-0.0931820870747269) * score 
            + math.pow(score, 2) * 0.00334709095418796 
            + math.pow(score, 3) * (-3.32988302362153E-05) 
            + math.pow(score, 4) * (1.75712910819776E-07) 
            + math.pow(score, 5) * (-5.59998691223273E-10) 
            + math.pow(score, 6) * (1.1296711294933E-12) 
            + math.pow(score, 7) * (-1.45218037113138e-15) 
            + math.pow(score, 8) * (1.15333377080353e-18) 
            + math.pow(score, 9) * (-5.15887481551927e-22) 
            + math.pow(score, 10) * (9.94098428102335e-26)
        )
    else:
        boneAge = (
            5.81191794824917 
            + (-0.271546561737745) * score 
            + math.pow(score, 2) * 0.00526301486340724 
            + math.pow(score, 3) * (-4.37797717401925E-05) 
            + math.pow(score, 4) * (2.0858722025667E-07) 
            + math.pow(score, 5) * (-6.21879866563429E-10) 
            + math.pow(score, 6) * (1.19909931745368E-12) 
            + math.pow(score, 7) * (-1.49462900826936E-15) 
            + math.pow(score, 8) * (1.162435538672E-18) 
            + math.pow(score, 9) * (-5.12713017846218E-22) 
            + math.pow(score, 10) * (9.78989966891478E-26)
        )
    
    return max(0, boneAge)


In [6]:
"""
数据集模块

包含:
- ArthrosisDataset: Arthritis关节分类数据集
- RSNADataset: RSNA骨龄数据集
"""

import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms


class ArthrosisDataset(Dataset):
    """Arthritis关节分类数据集"""
    
    def __init__(self, df: pd.DataFrame, transform=None, img_size: int = 256):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size
        
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        
        img_path = row['path']
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (self.img_size, self.img_size), (128, 128, 128))
        
        image = self.transform(image)
        
        result = {'image': image}
        
        if 'joint' in row and 'grade' in row:
            result['joint'] = row['joint']
            result['grade'] = torch.tensor(row['grade'] - 1, dtype=torch.long)
        
        if 'domain' in row:
            result['domain'] = torch.tensor(row['domain'], dtype=torch.long)
        
        return result


class RSNADataset(Dataset):
    """RSNA骨龄数据集"""
    
    def __init__(self, df: pd.DataFrame, img_dir: str, transform=None, img_size: int = 256):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.img_size = img_size
        
        self.age_min = df['boneage'].min()
        self.age_max = df['boneage'].max()
        
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        
        img_id = str(row['id'])
        img_path = os.path.join(self.img_dir, f"{img_id}.png")
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (self.img_size, self.img_size), (128, 128, 128))
        
        image = self.transform(image)
        
        bone_age = float(row['boneage'])
        gender = 1.0 if row['male'] else 0.0
        
        return {
            'image': image,
            'image_path': img_path,
            'bone_age': torch.tensor(bone_age, dtype=torch.float32),
            'gender': torch.tensor(gender, dtype=torch.float32)
        }


In [7]:
"""
数据处理工具模块

包含:
- split_arthrosis_dataset: 划分Arthritis数据集
- prepare_rsna_dataset: 准备RSNA数据集
"""

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from typing import Tuple


def split_arthrosis_dataset(data_root: str, 
                            train_ratio: float = 0.7, 
                            val_ratio: float = 0.15, 
                            test_ratio: float = 0.15,
                            default_grade: int = 1) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Arthritis数据集内部划分
    
    支持两种文件结构:
    1. 有等级子文件夹: /关节/等级/图片.jpg
    2. 无等级子文件夹: /关节/图片.jpg (使用default_grade作为等级)
    """
  
    
    all_samples = []
    
    actual_joints = []
    if os.path.exists(data_root):
        for joint_name in os.listdir(data_root):
            joint_path = os.path.join(data_root, joint_name)
            if os.path.isdir(joint_path):
                actual_joints.append(joint_name)
                print(f"  发现关节: {joint_name}")
    
    joint_names = actual_joints if actual_joints else Config.PHYSICAL_JOINT_NAMES
    print(f"  使用关节列表: {joint_names}")
    
    for joint_name in joint_names:
        joint_path = os.path.join(data_root, joint_name)
        if not os.path.exists(joint_path):
            continue
        
        subdirs = [d for d in os.listdir(joint_path) if os.path.isdir(os.path.join(joint_path, d))]
        grade_dirs = [d for d in subdirs if d.isdigit()]
        
        if grade_dirs:
            for grade in grade_dirs:
                grade_path = os.path.join(joint_path, grade)
                images = [f for f in os.listdir(grade_path) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                
                for img_name in images:
                    img_path = os.path.join(grade_path, img_name)
                    all_samples.append({
                        'path': img_path,
                        'joint': joint_name,
                        'grade': int(grade)
                    })
        else:
            images = [f for f in os.listdir(joint_path) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            for img_name in images:
                img_path = os.path.join(joint_path, img_name)
                all_samples.append({
                    'path': img_path,
                    'joint': joint_name,
                    'grade': default_grade
                })
    
    df = pd.DataFrame(all_samples)
    
    if len(df) == 0:
        print("  警告: 未找到任何训练数据！")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    print(f"  总样本数: {len(df)}")
    
    train_df, temp_df = train_test_split(df, test_size=(val_ratio + test_ratio), 
                                         stratify=df['joint'], random_state=42)
    
    val_test_ratio = val_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(temp_df, test_size=(1 - val_test_ratio),
                                      stratify=temp_df['joint'] if 'joint' in temp_df.columns and len(temp_df) > 0 else None,
                                      random_state=42)
    
    train_df['domain'] = 0
    val_df['domain'] = 0
    test_df['domain'] = 0
    
    return train_df, val_df, test_df


def prepare_rsna_dataset(csv_path: str, img_dir: str, num_samples: int = 10000) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """准备RSNA数据集"""
    
    if not os.path.exists(csv_path):
        print(f"警告: CSV文件不存在 {csv_path}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    df = pd.read_csv(csv_path)
    
    if len(df) > num_samples:
        df = df.sample(n=num_samples, random_state=42).reset_index(drop=True)
    
    boneage_bins = pd.cut(df['boneage'], bins=5, labels=False)
    train_df, temp_df = train_test_split(df, test_size=0.3, stratify=boneage_bins, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    train_df['domain'] = 1
    val_df['domain'] = 1
    test_df['domain'] = 1
    
    return train_df, val_df, test_df


In [8]:
"""
训练模块

包含:
- get_grl_lambda: GRL动态调度
- stage1_source_feature_learning: Stage1源域特征学习
- stage2_domain_adaptation: Stage2域自适应训练
- stage3_formula_bone_age: Stage3骨龄精调
"""

import torch
import torch.nn as nn
import pandas as pd
import math
import gc
import os
import copy
from typing import Dict, Tuple
from torch.utils.data import DataLoader
from torchvision import transforms, models
from tqdm import tqdm

import torch.nn.functional as F




HYPER_EDGES = {
    'finger_1': ['DIPFirst', 'PIPFirst', 'MCPFirst'],
    'finger_3': ['DIPThird', 'MIPThird', 'PIPThird', 'MCPThird'],
    'finger_5': ['DIPFifth', 'MIPFifth', 'PIPFifth', 'MCPFifth'],
    'forearm': ['Radius', 'Ulna'],
}

MIXUP_ALPHA = 0.0

def mixup_data(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = torch.distributions.Beta(alpha, alpha).sample()
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

HYPER_LOSS_WEIGHT = 0.1


def get_hyper_loss(logits_dict: dict, device: torch.device) -> torch.Tensor:
    """计算超图一致性损失：同组关节发育进度方差应较小"""
    h_loss = 0.0
    count = 0
    
    for edge_name, joints in HYPER_EDGES.items():
        group_logits = []
        for joint in joints:
            if joint in logits_dict and logits_dict[joint] is not None:
                group_logits.append(logits_dict[joint])
        
        if len(group_logits) > 1:
            probs_list = [F.softmax(log, dim=1) for log in group_logits]
            num_classes_list = [p.size(1) for p in probs_list]
            max_classes = max(num_classes_list)
            
            aligned_probs = []
            for i, probs in enumerate(probs_list):
                if probs.size(1) < max_classes:
                    pad = torch.zeros(probs.size(0), max_classes - probs.size(1), device=probs.device)
                    probs = torch.cat([probs, pad], dim=1)
                aligned_probs.append(probs)
            
            expected_grades = []
            for probs in aligned_probs:
                grade_indices = torch.arange(max_classes, device=device).float().unsqueeze(0)
                expected_grade = (probs * grade_indices).sum(dim=1)
                expected_grades.append(expected_grade)
            
            grades_tensor = torch.stack(expected_grades, dim=1)
            group_variance = torch.var(grades_tensor, dim=1).mean()
            h_loss += group_variance
            count += 1
    
    return h_loss / max(count, 1) if count > 0 else torch.tensor(0.0, device=device)


class MultiTaskJointModel(nn.Module):
    """多任务关节分类模型：共享骨干网络 + 独立分类头"""
    
    def __init__(self, joint_num_classes: dict, backbone='efficientnet_b0'):
        super().__init__()
        
        if backbone == 'resnet50':
            base_model = models.resnet50(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.features = nn.Sequential(*list(base_model.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 2048
        else:
            base_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.features = base_model.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 1280
        
        self.heads = nn.ModuleDict()
        for joint_name, num_classes in joint_num_classes.items():
            self.heads[joint_name] = nn.Linear(feat_dim, num_classes)
        
        self.feat_dim = feat_dim
    
    def forward(self, x: torch.Tensor) -> dict:
        """返回所有关节的logits字典"""
        feat = self.features(x)
        feat = self.pool(feat).flatten(1)
        
        logits_dict = {}
        for joint_name, head in self.heads.items():
            logits_dict[joint_name] = head(feat)
        
        return logits_dict
    
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """提取特征向量"""
        feat = self.features(x)
        return self.pool(feat).flatten(1)


def calc_bone_age_from_score_tensor(scores: torch.Tensor, genders: torch.Tensor) -> torch.Tensor:
    """
    可微分的骨龄计算（支持批量tensor），返回单位: 年。
    genders: 1.0=male, 0.0=female
    """
    return calc_bone_age_years_from_score_tensor(scores, genders)


def clear_gpu_memory():
    """清理GPU显存"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()


def _has_real_grade_labels(df: pd.DataFrame) -> bool:
    """判断数据是否包含真实等级标签而非默认占位标签。"""
    if df is None or len(df) == 0:
        return False

    if 'has_grade_label' in df.columns:
        return bool(df['has_grade_label'].fillna(False).any())

    if 'grade' not in df.columns:
        return False

    return df['grade'].nunique(dropna=True) > 1


def _rebalance_source_for_target(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_source_target_ratio: int = 8,
    min_source_samples: int = 32,
    random_state: int = 42,
) -> pd.DataFrame:
    """按目标域规模裁剪源域，避免极端不平衡破坏DANN训练。"""
    if source_df is None or len(source_df) == 0:
        return source_df
    if target_df is None or len(target_df) == 0:
        return source_df
    if max_source_target_ratio <= 0:
        return source_df

    target_size = len(target_df)
    capped_size = max(min_source_samples, target_size * max_source_target_ratio)
    capped_size = min(len(source_df), capped_size)

    if capped_size >= len(source_df):
        return source_df

    return source_df.sample(n=capped_size, random_state=random_state).reset_index(drop=True)


def get_num_classes_from_checkpoint(checkpoint: dict) -> int:
    """从checkpoint中自动检测类别数"""
    weight_keys = ['fc.weight', 'fc.1.weight', 'classifier.1.weight', 'classifier.weight',
                   'head_weight', 'label_classifier.weight']

    for key in weight_keys:
        if key in checkpoint:
            return checkpoint[key].shape[0]

    if 'num_classes' in checkpoint:
        return checkpoint['num_classes']

    for key in ['label_classifier.bias', 'domain_classifier.3.bias', 'fc.bias', 'fc.1.bias']:
        if key in checkpoint:
            return checkpoint[key].shape[0]

    available_keys = list(checkpoint.keys())

    if 'conv1.weight' in checkpoint or 'features.0.0.weight' in checkpoint:
        print(f"  ⚠️ 警告: 这是骨干网络权重文件，不包含分类层！")
        print(f"  可用键数量: {len(available_keys)}")
        print(f"  前20个键: {available_keys[:20]}")
        print(f"  可能原因: Stage1训练未完成或模型保存失败")

    raise KeyError(f"未找到类别权重键，可用键: {available_keys[:20]}... (共{len(available_keys)}个)")


def load_kaggle_model_for_stage2(model_path: str, device: torch.device) -> Tuple[nn.Module, int]:
    """
    加载Kaggle的Stage1预训练模型用于Stage2微调

    Args:
        model_path: Kaggle模型文件路径
        device: 计算设备

    Returns:
        dann_model: DomainAdversarialModel模型
        num_classes: 类别数
    """
    try:
        checkpoint = torch.load(model_path, weights_only=True)

        num_classes = get_num_classes_from_checkpoint(checkpoint)

        is_resnet = 'fc.1.weight' in checkpoint or 'fc.weight' in checkpoint

        if is_resnet:
            backbone = 'resnet50'
        else:
            backbone = 'efficientnet_b0'

        dann_model = DomainAdversarialModel(num_classes=num_classes, backbone=backbone)

        new_state = {}
        if 'fc.1.weight' in checkpoint:
            new_state['label_classifier.weight'] = checkpoint['fc.1.weight']
            new_state['label_classifier.bias'] = checkpoint['fc.1.bias']

            for k, v in checkpoint.items():
                if k.startswith('conv') or k.startswith('bn') or k.startswith('layer'):
                    new_state[k] = v
        elif 'fc.weight' in checkpoint:
            new_state['label_classifier.weight'] = checkpoint['fc.weight']
            new_state['label_classifier.bias'] = checkpoint['fc.bias']

            for k, v in checkpoint.items():
                if k.startswith('conv') or k.startswith('bn') or k.startswith('layer'):
                    new_state[k] = v

        dann_model.load_state_dict(new_state, strict=False)
        dann_model = dann_model.to(device)

        print(f"  ✅ Kaggle模型加载成功: {os.path.basename(model_path)}")
        print(f"     类别数: {num_classes}, 架构: {backbone}")

        return dann_model, num_classes

    except Exception as e:
        print(f"  ❌ Kaggle模型加载失败: {model_path}")
        print(f"     错误: {e}")
        raise


def get_grl_lambda(epoch: int, max_epochs: int) -> float:
    """GRL动态调度"""
    p = epoch / max_epochs
    return 2.0 / (1.0 + math.exp(-10 * p)) - 1.0


def stage1_source_feature_learning(source_train: pd.DataFrame,
                                    device: torch.device,
                                    num_epochs: int = 50,
                                    use_hypergraph: bool = True,
                                    val_ratio: float = 0.15,
                                    early_stop_patience: int = 20) -> Dict[str, str]:
    """Stage1: 每个物理关节训练一个独立模型
    
    Args:
        source_train: 源域训练数据
        device: 训练设备
        num_epochs: 训练轮数
        use_hypergraph: 是否使用超图约束（当前架构下不适用，将在Stage2多任务时启用）
        val_ratio: 验证集比例
        early_stop_patience: 早停耐心值
    """
    
    print("\n" + "=" * 60)
    print("        Stage1: 源域特征学习（一个关节一个模型）")
    print("=" * 60)
    
    if use_hypergraph:
        print(f"\n⚠ 注意: 超图约束需要在多任务学习架构下使用（Stage2）")
        print(f"  当前Stage1为单关节独立训练，超图损失将在后续Stage启用")
        print(f"  超图边定义:")
        for edge_name, joints in HYPER_EDGES.items():
            print(f"    - {edge_name}: {joints}")
        print(f"  超图损失权重: {HYPER_LOSS_WEIGHT}")
    
    trained_models = {}
    
    for physical_joint in Config.PHYSICAL_JOINT_NAMES:
        print(f"\n{'-'*60}")
        print(f"训练关节: {physical_joint}")
        print(f"{'-'*60}")
        
        joint_data = source_train[source_train['joint'] == physical_joint].copy()
        
        if len(joint_data) == 0:
            print(f"  跳过：没有{physical_joint}的数据")
            continue
        
        num_classes = joint_data['grade'].nunique()
        print(f"  数据量: {len(joint_data)}, 等级数: {num_classes}")
        
        from sklearn.model_selection import train_test_split
        train_data, val_data = train_test_split(
            joint_data, 
            test_size=val_ratio, 
            random_state=42, 
            stratify=joint_data['grade']
        )
        print(f"  训练集: {len(train_data)}, 验证集: {len(val_data)}")
        
        img_size = Config.IMG_SIZE
        
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        val_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        train_dataset = ArthrosisDataset(train_data, transform=train_transform)
        val_dataset = ArthrosisDataset(val_data, transform=val_transform)
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
        
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(2048, num_classes)
        )
        model = model.to(device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        ce_loss = nn.CrossEntropyLoss()
        
        best_val_acc = 0.0
        best_model_state = None
        patience_counter = 0
        
        for epoch in range(num_epochs):
            torch.cuda.empty_cache()
            
            model.train()
            train_loss = 0
            train_count = 0
            train_correct = 0
            train_total = 0
            
            pbar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{num_epochs}")
            for batch in pbar:
                images = batch['image'].to(device)
                grades = batch['grade'].to(device)
                
                mixed_images, y_a, y_b, lam = mixup_data(images, grades, MIXUP_ALPHA)
                logits = model(mixed_images)
                loss = mixup_criterion(ce_loss, logits, y_a, y_b, lam)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * images.size(0)
                train_count += images.size(0)
                
                _, predicted = torch.max(logits, 1)
                train_total += grades.size(0)
                train_correct += (predicted == grades).sum().item()
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
                
                del mixed_images, y_a, y_b, lam, logits, loss
            
            train_loss = train_loss / max(train_count, 1)
            train_acc = 100.0 * train_correct / max(train_total, 1)
            
            model.eval()
            val_loss = 0
            val_count = 0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for batch in val_loader:
                    images = batch['image'].to(device)
                    grades = batch['grade'].to(device)
                    
                    logits = model(images)
                    loss = ce_loss(logits, grades)
                    
                    val_loss += loss.item() * images.size(0)
                    val_count += images.size(0)
                    
                    _, predicted = torch.max(logits, 1)
                    val_total += grades.size(0)
                    val_correct += (predicted == grades).sum().item()
            
            val_loss = val_loss / max(val_count, 1)
            val_acc = 100.0 * val_correct / max(val_total, 1)
            
            scheduler.step(-val_acc)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
                marker = " ✓"
            else:
                patience_counter += 1
                marker = ""
            
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1}/{num_epochs}: train_loss={train_loss:.4f}, train_acc={train_acc:.1f}%, "
                      f"val_loss={val_loss:.4f}, val_acc={val_acc:.1f}%, "
                      f"lr={optimizer.param_groups[0]['lr']:.2e}{marker}")
            
            if patience_counter >= early_stop_patience:
                print(f"  早停触发于 epoch {epoch+1}，最佳验证精度: {best_val_acc:.2f}%")
                break
        
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        model_path = f'stage1_{physical_joint}_model.pth'
        torch.save(model.state_dict(), model_path)
        trained_models[physical_joint] = model_path
        print(f"  ✓ {physical_joint}模型已保存: {model_path} (best_val_acc={best_val_acc:.2f}%)")
        
        del model, best_model_state
        clear_gpu_memory()
    
    print(f"\n{'='*60}")
    print(f"Stage1完成！共训练 {len(trained_models)} 个关节模型")
    print(f"{'='*60}")
    
    return trained_models


class DomainAdversarialModel(nn.Module):
    """DANN模型：特征提取器 + 标签分类器 + 域分类器（GRL反转）"""
    
    def __init__(self, num_classes: int, backbone='efficientnet_b0'):
        super().__init__()
        
        if backbone == 'resnet50':
            base_model = models.resnet50(weights=None)
            self.features = nn.Sequential(*list(base_model.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 2048
        else:
            base_model = models.efficientnet_b0(weights=None)
            self.features = base_model.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 1280
        
        self.label_classifier = nn.Linear(feat_dim, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        
        self.feat_dim = feat_dim
    
    def forward(self, x: torch.Tensor, alpha: float = 1.0) -> Tuple[torch.Tensor, torch.Tensor]:
        """前向传播，同时输出标签预测和域预测"""
        feat = self.features(x)
        feat = self.pool(feat).flatten(1)
        
        class_logits = self.label_classifier(feat)
        
        reversed_feat = GradientReversalFunction.apply(feat, alpha)
        domain_logits = self.domain_classifier(reversed_feat)
        
        return class_logits, domain_logits
    
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """提取特征向量（不包含GRL）"""
        feat = self.features(x)
        feat = self.pool(feat).flatten(1)
        return feat


class MultiTaskDANNModel(nn.Module):
    """多任务DANN模型：支持超图约束的多关节域自适应"""
    
    def __init__(self, joint_num_classes: dict, backbone='efficientnet_b0'):
        super().__init__()
        
        if backbone == 'resnet50':
            base_model = models.resnet50(weights=None)
            self.features = nn.Sequential(*list(base_model.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 2048
        else:
            base_model = models.efficientnet_b0(weights=None)
            self.features = base_model.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            feat_dim = 1280
        
        self.label_classifiers = nn.ModuleDict()
        for joint_name, num_classes in joint_num_classes.items():
            self.label_classifiers[joint_name] = nn.Linear(feat_dim, num_classes)
        
        self.domain_classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        
        self.feat_dim = feat_dim
        self.joint_names = list(joint_num_classes.keys())
    
    def forward(self, x: torch.Tensor, alpha: float = 1.0, joint_names: list = None) -> Tuple[dict, torch.Tensor]:
        """前向传播，返回指定关节的logits和域logits"""
        feat = self.features(x)
        feat = self.pool(feat).flatten(1)
        
        if joint_names is None:
            joint_names = self.joint_names
        
        class_logits_dict = {}
        for joint_name in joint_names:
            if joint_name in self.label_classifiers:
                class_logits_dict[joint_name] = self.label_classifiers[joint_name](feat)
        
        reversed_feat = GradientReversalFunction.apply(feat, alpha)
        domain_logits = self.domain_classifier(reversed_feat)
        
        return class_logits_dict, domain_logits
    
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """提取特征向量（不包含GRL）"""
        feat = self.features(x)
        return self.pool(feat).flatten(1)


def gaussian_kernel(x: torch.Tensor, y: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """计算高斯核矩阵"""
    n = x.size(0)
    m = y.size(0)
    
    x = x.unsqueeze(1).expand(n, m, -1)
    y = y.unsqueeze(0).expand(n, m, -1)
    
    kernel = torch.exp(-torch.sum((x - y) ** 2, dim=2) / (2 * sigma ** 2))
    return kernel


def compute_mmd(source_feat: torch.Tensor, target_feat: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """计算MMD损失"""
    K_ss = gaussian_kernel(source_feat, source_feat, sigma).mean()
    K_tt = gaussian_kernel(target_feat, target_feat, sigma).mean()
    K_st = gaussian_kernel(source_feat, target_feat, sigma).mean()
    
    mmd = K_ss + K_tt - 2 * K_st
    return torch.clamp(mmd, min=0)


class RandomBatchSampler:
    """随机批次采样器，用于生成源域和目标域的配对批次"""
    def __init__(self, source_loader: DataLoader, target_loader: DataLoader):
        self.source_loader = source_loader
        self.target_loader = target_loader
        self.num_batches = max(len(source_loader), len(target_loader))
        self.reset_iterators()
    
    def __iter__(self):
        self.batch_count = 0
        return self
    
    def __next__(self):
        if self.batch_count >= self.num_batches:
            raise StopIteration

        self.batch_count += 1

        try:
            source_batch = next(self.source_iter)
        except StopIteration:
            self.source_iter = iter(self.source_loader)
            source_batch = next(self.source_iter)
        
        try:
            target_batch = next(self.target_iter)
        except StopIteration:
            self.target_iter = iter(self.target_loader)
            target_batch = next(self.target_iter)
        
        return source_batch, target_batch

    def __len__(self):
        return self.num_batches

    def reset_iterators(self):
        self.source_iter = iter(self.source_loader)
        self.target_iter = iter(self.target_loader)
        self.batch_count = 0


class HypergraphBatchSampler:
    """超图批次采样器：每个batch从同一超图组中采样多个关节"""
    def __init__(self, loaders_dict: Dict[str, DataLoader], hyper_edges: dict):
        self.loaders_dict = loaders_dict
        self.hyper_edges = hyper_edges
        self.joint_names = list(loaders_dict.keys())
        self.num_batches = max((len(loader) for loader in loaders_dict.values()), default=0)
        self.reset_iterators()
    
    def __iter__(self):
        self.batch_count = 0
        return self
    
    def __next__(self):
        if self.batch_count >= self.num_batches:
            raise StopIteration

        self.batch_count += 1

        batches_by_edge = {}
        for edge_name, joints in self.hyper_edges.items():
            edge_batches = {}
            for joint in joints:
                if joint not in self.loaders_dict:
                    continue
                try:
                    edge_batches[joint] = next(self.iterators[joint])
                except StopIteration:
                    self.iterators[joint] = iter(self.loaders_dict[joint])
                    edge_batches[joint] = next(self.iterators[joint])
            if len(edge_batches) > 0:
                batches_by_edge[edge_name] = edge_batches
        return batches_by_edge
    
    def __len__(self):
        return self.num_batches

    def reset_iterators(self):
        self.iterators = {name: iter(loader) for name, loader in self.loaders_dict.items()}
        self.batch_count = 0


def stage2_domain_adaptation(stage1_models: Dict[str, str],
                            target_train: pd.DataFrame,
                            device: torch.device,
                            num_epochs: int = 5,
                            batch_size: int = 16,
                            train_transform=None,
                            source_train: pd.DataFrame = None,
                            training_mode: str = 'auto',
                            max_source_target_ratio: int = 8,
                            min_source_samples: int = 32) -> Dict[str, str]:
    """Stage2: 完整DANN域自适应训练
    
    Args:
        stage1_models: Stage1训练好的模型路径字典
        target_train: 目标域训练数据
        device: 训练设备
        num_epochs: 训练轮数
        batch_size: 批大小
        train_transform: 数据增强
        source_train: 源域训练数据（新增，用于DANN）
        training_mode: 训练模式，支持 auto / balanced_dann
        max_source_target_ratio: balanced_dann 时每个关节源域相对目标域的最大倍率
        min_source_samples: balanced_dann 时每个关节保留的最小源域样本数
    """
    
    print("\n" + "=" * 60)
    print("        Stage2: DANN域自适应训练（完整实现）")
    print("=" * 60)
    
    print(f"\n目标域训练数据量: {len(target_train)}")
    source_train_size = len(source_train) if source_train is not None else 0
    print(f"源域训练数据量: {source_train_size}")
    
    if len(stage1_models) == 0:
        print("  警告: Stage1模型为空，跳过域自适应训练")
        return {}
    
    if len(target_train) == 0:
        print("  警告: 目标域训练数据为空，跳过域自适应训练")
        return stage1_models
    
    has_source_train = source_train is not None and len(source_train) > 0
    if not has_source_train:
        print("  警告: 源域训练数据为空，将按关节回退到目标域监督微调或保留Stage1模型")
    else:
        print("  ✓ 源域数据已加载，将使用DANN对抗训练")
    
    trained_models = {}
    
    available_models = list(stage1_models.keys())
    print(f"\n  可用模型: {available_models}")
    print(f"  将训练 {len(available_models)} 个关节的DANN模型")
    
    for physical_joint in available_models:
        stage1_path = stage1_models[physical_joint]
        
        print(f"\n{'-'*60}")
        print(f"DANN域自适应训练: {physical_joint}")
        print(f"{'-'*60}")
        
        try:
            checkpoint = torch.load(stage1_path, weights_only=True)
            
            keys = list(checkpoint.keys())
            is_resnet = 'conv1.weight' in checkpoint and any('layer1' in k for k in keys)
            is_efficientnet = 'features.0.0.weight' in checkpoint or 'features.2.0.weight' in checkpoint
            
            has_fc = 'fc.weight' in checkpoint or 'fc.0.weight' in checkpoint or 'fc.1.weight' in checkpoint or 'classifier.1.weight' in checkpoint
            
            if has_fc:
                if 'fc.weight' in checkpoint:
                    num_classes = checkpoint['fc.weight'].shape[0]
                elif 'fc.1.weight' in checkpoint:
                    num_classes = checkpoint['fc.1.weight'].shape[0]
                elif 'fc.0.weight' in checkpoint:
                    num_classes = checkpoint['fc.0.weight'].shape[0]
                elif 'classifier.1.weight' in checkpoint:
                    num_classes = checkpoint['classifier.1.weight'].shape[0]
                else:
                    num_classes = Config.MEDICAL_JOINT_NUM_CLASSES.get(physical_joint, 3)
                    print(f"  ⚠️ 未找到预训练分类头，使用默认类别数: {num_classes}")
            else:
                num_classes = Config.MEDICAL_JOINT_NUM_CLASSES.get(physical_joint, 3)
                print(f"  ⚠️ 未找到预训练分类头，使用默认类别数: {num_classes}")
            
            if is_resnet:
                backbone = 'resnet50'
            elif is_efficientnet:
                backbone = 'efficientnet_b0'
            else:
                backbone = 'resnet50'
            
            dann_model = DomainAdversarialModel(num_classes=num_classes, backbone=backbone)
            
            new_state = {}
            
            if has_fc:
                if 'fc.weight' in checkpoint:
                    new_state['label_classifier.weight'] = checkpoint['fc.weight']
                    new_state['label_classifier.bias'] = checkpoint['fc.bias']
                elif 'fc.1.weight' in checkpoint:
                    new_state['label_classifier.weight'] = checkpoint['fc.1.weight']
                    new_state['label_classifier.bias'] = checkpoint['fc.1.bias']
                elif 'fc.0.weight' in checkpoint:
                    new_state['label_classifier.weight'] = checkpoint['fc.0.weight']
                    new_state['label_classifier.bias'] = checkpoint['fc.0.bias']
                elif 'classifier.1.weight' in checkpoint:
                    new_state['label_classifier.weight'] = checkpoint['classifier.1.weight']
                    new_state['label_classifier.bias'] = checkpoint['classifier.1.bias']
                
                for k, v in checkpoint.items():
                    if k.startswith('conv') or k.startswith('bn') or k.startswith('layer') or k.startswith('features'):
                        new_state[k] = v
            
            dann_model.load_state_dict(new_state, strict=False)
            dann_model = dann_model.to(device)
            
            print(f"  已加载Stage1模型: {stage1_path}")
            print(f"  模型架构: {backbone} + DANN")
            if has_fc:
                print(f"  包含预训练分类头: num_classes={num_classes}")
            else:
                print(f"  仅backbone权重，将训练新的分类头 (num_classes={num_classes})")
        except Exception as e:
            print(f"  错误: 加载Stage1模型失败 {stage1_path}: {e}")
            continue
        
        target_joint_data = target_train[target_train['joint'] == physical_joint].copy()
        if len(target_joint_data) == 0:
            print(f"  ⚠️ 跳过 {physical_joint}: 目标域无数据")
            print(f"     使用原始Kaggle模型（未经过域自适应训练）")
            trained_models[physical_joint] = stage1_path
            continue
        
        print(f"  目标域数据量: {len(target_joint_data)}")

        target_has_labels = _has_real_grade_labels(target_joint_data)
        source_joint_data = pd.DataFrame()
        if has_source_train:
            source_joint_data = source_train[source_train['joint'] == physical_joint].copy()

        if training_mode == 'balanced_dann' and len(source_joint_data) > 0:
            original_source_size = len(source_joint_data)
            source_joint_data = _rebalance_source_for_target(
                source_joint_data,
                target_joint_data,
                max_source_target_ratio=max_source_target_ratio,
                min_source_samples=min_source_samples,
            )
            if len(source_joint_data) != original_source_size:
                print(
                    f"  ✓ 平衡DANN采样: 源域样本 {original_source_size} -> {len(source_joint_data)} "
                    f"(目标域 {len(target_joint_data)}, 倍率上限 {max_source_target_ratio}x, "
                    f"最小保留 {min_source_samples})"
                )

        joint_use_dann = len(source_joint_data) > 0
        joint_use_target_supervision = (not joint_use_dann) and target_has_labels
        joint_use_mixed_dann = (not joint_use_dann) and (not joint_use_target_supervision) and len(target_joint_data) > 0 and has_source_train

        target_dataset = ArthrosisDataset(target_joint_data, transform=train_transform)
        target_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=True)

        if joint_use_dann:
            print(f"  源域数据量: {len(source_joint_data)}")
            print("  训练模式: 源域分类监督 + DANN域对抗 + MMD对齐")
            source_dataset = ArthrosisDataset(source_joint_data, transform=train_transform)
            source_loader = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)
            batch_sampler = RandomBatchSampler(source_loader, target_loader)
        elif joint_use_target_supervision:
            print("  训练模式: 目标域监督微调（无源域配对数据）")
        elif joint_use_mixed_dann:
            print(f"  目标域数据量: {len(target_joint_data)}")
            print("  训练模式: 混合DANN（源域+目标域无标签混合训练）")
            mixed_data = pd.concat([source_joint_data, target_joint_data], ignore_index=True)
            mixed_dataset = ArthrosisDataset(mixed_data, transform=train_transform)
            mixed_loader = DataLoader(mixed_dataset, batch_size=batch_size, shuffle=True)
            source_joint_data_for_dann = source_joint_data
        else:
            print("  警告: 当前关节缺少源域配对数据，且目标域无真实等级标签")
            print("     保留Stage1模型，不执行无效的域分类训练")
            trained_models[physical_joint] = stage1_path
            del dann_model, checkpoint
            clear_gpu_memory()
            continue
        
        dann_optimizer = torch.optim.AdamW(dann_model.parameters(), lr=1e-5, weight_decay=1e-4)
        ce_loss_fn = nn.CrossEntropyLoss()
        
        print("\n  开始Stage2训练...")
        if joint_use_dann:
            print("  训练策略: 最大化源域标签分类精度 + 最小化域分类精度（GRL反转梯度）")
        elif joint_use_mixed_dann:
            print("  训练策略: 混合DANN（源域分类监督 + GRL域对抗）")
        else:
            print("  训练策略: 使用目标域真实等级标签进行监督微调")
        
        target_data_size = len(target_joint_data)
        mmd_weight = 0.01 if target_data_size >= 100 else 0.001
        
        if target_data_size < 50:
            print(f"  ⚠️ 警告: 目标域数据量较少({target_data_size})，降低MMD权重至{mmd_weight}以避免过拟合")
        elif target_data_size >= 100:
            print(f"  ✓ 目标域数据量充足({target_data_size})，使用标准MMD权重")
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            dann_optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
        )
        
        best_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        early_stop_patience = 5
        best_model_saved = False
        
        for epoch in range(num_epochs):
            dann_model.train()
            
            alpha = get_grl_lambda(epoch, num_epochs)
            
            total_class_loss = 0
            total_domain_loss = 0
            total_mmd_loss = 0
            total_objective_loss = 0
            total_count = 0
            
            if joint_use_dann:
                batch_sampler.reset_iterators()
                pbar = tqdm(batch_sampler, total=len(batch_sampler), desc=f"  Epoch {epoch+1}/{num_epochs} [α={alpha:.3f}]")
                for source_batch, target_batch in pbar:
                    source_images = source_batch['image'].to(device)
                    source_labels = source_batch['grade'].to(device)
                    target_images = target_batch['image'].to(device)
                    
                    dann_optimizer.zero_grad()
                    
                    source_class_logits, source_domain_logits = dann_model(source_images, alpha=alpha)
                    _, target_domain_logits = dann_model(target_images, alpha=alpha)

                    class_loss = ce_loss_fn(source_class_logits, source_labels)
                    
                    domain_loss = ce_loss_fn(source_domain_logits, torch.zeros(source_domain_logits.size(0), dtype=torch.long, device=device))
                    domain_loss += ce_loss_fn(target_domain_logits, torch.ones(target_domain_logits.size(0), dtype=torch.long, device=device))
                    domain_loss = domain_loss / 2
                    
                    source_feats = dann_model.extract_features(source_images)
                    target_feats = dann_model.extract_features(target_images)
                    mmd = compute_mmd(source_feats, target_feats)
                    
                    total_loss = class_loss + 0.3 * domain_loss + mmd_weight * mmd
                    
                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(dann_model.parameters(), max_norm=1.0)
                    dann_optimizer.step()
                    
                    total_class_loss += class_loss.item()
                    total_domain_loss += domain_loss.item()
                    total_mmd_loss += mmd.item()
                    total_objective_loss += total_loss.item()
                    total_count += 1
                    
                    pbar.set_postfix({
                        'cls': f'{class_loss.item():.4f}',
                        'dom': f'{domain_loss.item():.4f}',
                        'mmd': f'{mmd.item():.4f}',
                        'obj': f'{total_loss.item():.4f}',
                    })
            elif joint_use_mixed_dann:
                pbar = tqdm(mixed_loader, desc=f"  Epoch {epoch+1}/{num_epochs} [α={alpha:.3f}]")
                for batch in pbar:
                    images = batch['image'].to(device)
                    grades = batch['grade'].to(device)

                    dann_optimizer.zero_grad()

                    class_logits, domain_logits = dann_model(images, alpha=alpha)

                    class_loss = ce_loss_fn(class_logits, grades)

                    domain_loss = ce_loss_fn(domain_logits, torch.zeros(domain_logits.size(0), dtype=torch.long, device=device))

                    total_loss = class_loss + 0.3 * domain_loss

                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(dann_model.parameters(), max_norm=1.0)
                    dann_optimizer.step()

                    total_class_loss += class_loss.item()
                    total_domain_loss += domain_loss.item()
                    total_objective_loss += total_loss.item()
                    total_count += 1

                    pbar.set_postfix({
                        'cls': f'{class_loss.item():.4f}',
                        'dom': f'{domain_loss.item():.4f}',
                    })
            else:
                pbar = tqdm(target_loader, desc=f"  Epoch {epoch+1}/{num_epochs}")
                for batch in pbar:
                    images = batch['image'].to(device)
                    grades = batch['grade'].to(device)
                    
                    dann_optimizer.zero_grad()
                    
                    class_logits, _ = dann_model(images, alpha=0.0)
                    loss = ce_loss_fn(class_logits, grades)
                    
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(dann_model.parameters(), max_norm=1.0)
                    dann_optimizer.step()
                    
                    total_class_loss += loss.item()
                    total_objective_loss += loss.item()
                    total_count += 1
                    pbar.set_postfix({'cls': f'{loss.item():.4f}'})
            
            avg_class_loss = total_class_loss / max(total_count, 1)
            avg_domain_loss_total = total_domain_loss / max(total_count, 1)
            avg_mmd_loss = total_mmd_loss / max(total_count, 1)
            avg_objective_loss = total_objective_loss / max(total_count, 1)
            scheduler.step(avg_objective_loss)
            
            current_lr = dann_optimizer.param_groups[0]['lr']
            
            if joint_use_dann:
                print(f"    Epoch {epoch+1}/{num_epochs} [LR={current_lr:.2e}]: "
                      f"分类={avg_class_loss:.4f}, "
                      f"域={avg_domain_loss_total:.4f}, "
                      f"MMD={avg_mmd_loss:.4f}, "
                      f"总损失={avg_objective_loss:.4f}")

                if avg_objective_loss < best_loss:
                    best_loss = avg_objective_loss
                    patience_counter = 0
                    best_model_state = copy.deepcopy(dann_model.state_dict())
                    best_model_saved = False
                else:
                    patience_counter += 1
                    if patience_counter >= early_stop_patience:
                        print(f"    ⚠️ 早停触发: 连续{early_stop_patience}个epoch无改善，停止训练")
                        if best_model_state is not None:
                            dann_model.load_state_dict(best_model_state)
                        break
            elif joint_use_mixed_dann:
                print(f"    Epoch {epoch+1}/{num_epochs} [LR={current_lr:.2e}]: "
                      f"分类={avg_class_loss:.4f}, "
                      f"域={avg_domain_loss_total:.4f}, "
                      f"总损失={avg_objective_loss:.4f}")

                if avg_objective_loss < best_loss:
                    best_loss = avg_objective_loss
                    patience_counter = 0
                    best_model_state = copy.deepcopy(dann_model.state_dict())
                    best_model_saved = False
                else:
                    patience_counter += 1
                    if patience_counter >= early_stop_patience:
                        print(f"    ⚠️ 早停触发: 连续{early_stop_patience}个epoch无改善，停止训练")
                        if best_model_state is not None:
                            dann_model.load_state_dict(best_model_state)
                        break
            else:
                print(f"    Epoch {epoch+1}/{num_epochs} [LR={current_lr:.2e}]: "
                      f"分类={avg_class_loss:.4f}, "
                      f"总损失={avg_objective_loss:.4f}")
                
                if avg_objective_loss < best_loss:
                    best_loss = avg_objective_loss
                    patience_counter = 0
                    best_model_state = copy.deepcopy(dann_model.state_dict())
                    best_model_saved = False
                else:
                    patience_counter += 1
                    if patience_counter >= early_stop_patience:
                        print(f"    ⚠️ 早停触发: 连续{early_stop_patience}个epoch无改善，停止训练")
                        if best_model_state is not None:
                            dann_model.load_state_dict(best_model_state)
                        break
        
        if best_model_state is not None:
            dann_model.load_state_dict(best_model_state)
            if not best_model_saved:
                best_stage2_path = f'stage2_{physical_joint}_best_domain_loss_model.pth'
                torch.save(best_model_state, best_stage2_path)
                best_model_saved = True
                print(f"  ✓ 最佳域损失模型已保存: {best_stage2_path}")
        
        stage2_path = f'stage2_{physical_joint}_model.pth'
        torch.save(dann_model.state_dict(), stage2_path)
        trained_models[physical_joint] = stage2_path
        print(f"  ✓ Stage2模型已保存: {stage2_path} (best_loss={best_loss:.4f})")
        
        del dann_model, checkpoint
        clear_gpu_memory()
        print(f"  ✓ {physical_joint}显存已释放")
    
    print(f"\n{'='*60}")
    print(f"Stage2 DANN域自适应训练完成！共 {len(trained_models)} 个模型")
    print(f"{'='*60}")
    
    return trained_models


def stage2_hypergraph_dann(stage1_models: Dict[str, str],
                           target_train: pd.DataFrame,
                           device: torch.device,
                           num_epochs: int = 5,
                           batch_size: int = 8,
                           train_transform=None,
                           source_train: pd.DataFrame = None,
                           hypergraph_weight: float = 0.1) -> Dict[str, str]:
    """Stage2: 支持超图约束的多任务DANN域自适应训练
    
    Args:
        stage1_models: Stage1训练好的模型路径字典
        target_train: 目标域训练数据
        device: 训练设备
        num_epochs: 训练轮数
        batch_size: 批大小（每个关节的batch）
        train_transform: 数据增强
        source_train: 源域训练数据
        hypergraph_weight: 超图损失权重
    """
    
    print("\n" + "=" * 60)
    print("   Stage2: 超图感知的DANN域自适应训练")
    print("=" * 60)
    
    print(f"\n目标域训练数据量: {len(target_train)}")
    print(f"源域训练数据量: {len(source_train) if source_train is not None else 0}")
    print(f"超图损失权重: {hypergraph_weight}")
    print(f"\n超图边定义:")
    for edge_name, joints in HYPER_EDGES.items():
        print(f"  - {edge_name}: {joints}")
    
    if len(stage1_models) == 0:
        print("  警告: Stage1模型为空，跳过域自适应训练")
        return {}
    
    if len(target_train) == 0:
        print("  警告: 目标域训练数据为空，跳过域自适应训练")
        return stage1_models
    
    use_dann = source_train is not None and len(source_train) > 0
    if use_dann:
        print("  ✓ 源域数据已加载，将使用DANN对抗训练")
    else:
        print("  警告: 源域训练数据为空，使用传统迁移学习方法")
    
    available_joints = [j for j in Config.PHYSICAL_JOINT_NAMES if j in stage1_models]
    print(f"\n可用关节数: {len(available_joints)}")
    
    joint_num_classes = {}
    for physical_joint in available_joints:
        stage1_path = stage1_models[physical_joint]
        checkpoint = torch.load(stage1_path, weights_only=True)
        joint_num_classes[physical_joint] = get_num_classes_from_checkpoint(checkpoint)
    
    backbone = 'resnet50'
    print(f"  模型架构: {backbone} (多任务DANN + 超图约束)")
    
    mult_task_model = MultiTaskDANNModel(joint_num_classes=joint_num_classes, backbone=backbone)
    
    for physical_joint in available_joints:
        stage1_path = stage1_models[physical_joint]
        checkpoint = torch.load(stage1_path, weights_only=True)
        
        if 'classifier.weight' in checkpoint:
            mult_task_model.label_classifiers[physical_joint].weight.data = checkpoint['classifier.weight']
            mult_task_model.label_classifiers[physical_joint].bias.data = checkpoint['classifier.bias']
        elif 'fc.weight' in checkpoint:
            mult_task_model.label_classifiers[physical_joint].weight.data = checkpoint['fc.weight']
            mult_task_model.label_classifiers[physical_joint].bias.data = checkpoint['fc.bias']
        
        if 'features.1.weight' in checkpoint or 'classifier.1.weight' in checkpoint:
            for k, v in checkpoint.items():
                if k.startswith('features.'):
                    set_k = k.replace('features.', 'features.')
                    if hasattr(mult_task_model.features, k.split('.')[1]):
                        getattr(mult_task_model.features, k.split('.')[1]).weight.data = v
    
    mult_task_model = mult_task_model.to(device)
    print(f"  ✓ 多任务DANN模型已创建，共 {len(available_joints)} 个关节")
    
    target_loaders = {}
    source_loaders = {}
    
    for physical_joint in available_joints:
        target_joint_data = target_train[target_train['joint'] == physical_joint].copy()
        if len(target_joint_data) > 0:
            target_dataset = ArthrosisDataset(target_joint_data, transform=train_transform)
            target_loaders[physical_joint] = DataLoader(target_dataset, batch_size=batch_size, shuffle=True)
        
        if use_dann and source_train is not None:
            source_joint_data = source_train[source_train['joint'] == physical_joint].copy()
            if len(source_joint_data) > 0:
                source_dataset = ArthrosisDataset(source_joint_data, transform=train_transform)
                source_loaders[physical_joint] = DataLoader(source_dataset, batch_size=batch_size, shuffle=True)
    
    target_data_size = sum(len(loader.dataset) for loader in target_loaders.values())
    mmd_weight = 0.01 if target_data_size >= 100 else 0.001
    print(f"\n目标域数据总量: {target_data_size}, MMD权重: {mmd_weight}")
    
    hyper_sampler = HypergraphBatchSampler(target_loaders, HYPER_EDGES)
    
    optimizer = torch.optim.AdamW(mult_task_model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )
    ce_loss_fn = nn.CrossEntropyLoss()
    
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    early_stop_patience = 5
    best_model_saved = False
    
    print("\n  开始超图感知DANN对抗训练...")
    print("  训练策略: 标签分类 + 域自适应 + 超图一致性约束")
    
    for epoch in range(num_epochs):
        mult_task_model.train()
        
        alpha = get_grl_lambda(epoch, num_epochs)
        
        total_domain_loss = 0
        total_mmd_loss = 0
        total_hyper_loss = 0
        total_count = 0
        
        if use_dann and len(source_loaders) > 0:
            source_hyper_sampler = HypergraphBatchSampler(source_loaders, HYPER_EDGES)
            
            pbar = tqdm(range(len(hyper_sampler)), desc=f"  Epoch {epoch+1}/{num_epochs} [α={alpha:.3f}]")
            
            for batch_idx in pbar:
                try:
                    source_batches = next(source_hyper_sampler)
                    target_batches = next(hyper_sampler)
                except StopIteration:
                    hyper_sampler.reset_iterators()
                    source_hyper_sampler.reset_iterators()
                    break
                
                optimizer.zero_grad()
                
                batch_logits = {}
                batch_domain_loss = 0
                batch_mmd_loss = 0
                batch_hyper_loss = 0
                batch_class_loss = 0
                
                for edge_name in HYPER_EDGES.keys():
                    if edge_name not in source_batches or edge_name not in target_batches:
                        continue
                    
                    edge_source_batches = source_batches[edge_name]
                    edge_target_batches = target_batches[edge_name]
                    
                    all_images = []
                    all_joints = []
                    
                    for joint_name, batch in edge_source_batches.items():
                        all_images.append(batch['image'].to(device))
                        all_joints.append(joint_name)
                    
                    for joint_name, batch in edge_target_batches.items():
                        all_images.append(batch['image'].to(device))
                        all_joints.append(joint_name)
                    
                    if len(all_images) == 0:
                        continue
                    
                    all_images_cat = torch.cat(all_images, dim=0)
                    
                    class_logits_dict, domain_logits = mult_task_model(
                        all_images_cat, alpha=alpha, joint_names=all_joints
                    )
                    
                    source_size = len(edge_source_batches)
                    source_domain = torch.zeros(source_size, dtype=torch.long, device=device)
                    target_domain = torch.ones(len(all_images_cat) - source_size, dtype=torch.long, device=device)
                    domain_labels = torch.cat([source_domain, target_domain])
                    domain_loss = ce_loss_fn(domain_logits, domain_labels)
                    
                    source_images = torch.cat([batch['image'].to(device) for batch in edge_source_batches.values()])
                    target_images = torch.cat([batch['image'].to(device) for batch in edge_target_batches.values()])
                    
                    source_feats = mult_task_model.extract_features(source_images)
                    target_feats = mult_task_model.extract_features(target_images)
                    mmd = compute_mmd(source_feats, target_feats)
                    
                    edge_logits = {k: v[:len(edge_source_batches.get(k, [None]) * batch_size)] 
                                   for k, v in class_logits_dict.items() if k in edge_source_batches}
                    
                    if len(edge_logits) > 1:
                        hyper_loss = get_hyper_loss(edge_logits, device)
                    else:
                        hyper_loss = torch.tensor(0.0, device=device)
                    
                    total_loss = 0.3 * domain_loss + mmd_weight * mmd + hypergraph_weight * hyper_loss
                    
                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(mult_task_model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    total_domain_loss += domain_loss.item()
                    total_mmd_loss += mmd.item()
                    total_hyper_loss += hyper_loss.item()
                    total_count += 1
                
                pbar.set_postfix({
                    'dom': f'{total_domain_loss/max(total_count,1):.4f}',
                    'mmd': f'{total_mmd_loss/max(total_count,1):.4f}',
                    'hyp': f'{total_hyper_loss/max(total_count,1):.4f}'
                })
        else:
            pbar = tqdm(range(len(hyper_sampler)), desc=f"  Epoch {epoch+1}/{num_epochs}")
            
            for batch_idx in pbar:
                try:
                    target_batches = next(hyper_sampler)
                except StopIteration:
                    hyper_sampler.reset_iterators()
                    break
                
                optimizer.zero_grad()
                
                for edge_name in HYPER_EDGES.keys():
                    if edge_name not in target_batches:
                        continue
                    
                    edge_target_batches = target_batches[edge_name]
                    
                    all_images = []
                    all_joints = []
                    
                    for joint_name, batch in edge_target_batches.items():
                        all_images.append(batch['image'].to(device))
                        all_joints.append(joint_name)
                    
                    if len(all_images) == 0:
                        continue
                    
                    all_images_cat = torch.cat(all_images, dim=0)
                    
                    class_logits_dict, _ = mult_task_model(
                        all_images_cat, alpha=0, joint_names=all_joints
                    )
                    
                    edge_logits = {k: v for k, v in class_logits_dict.items() if k in edge_target_batches}
                    if len(edge_logits) > 1:
                        hyper_loss = get_hyper_loss(edge_logits, device)
                    else:
                        hyper_loss = torch.tensor(0.0, device=device)
                    
                    total_loss = hypergraph_weight * hyper_loss
                    
                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(mult_task_model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    total_hyper_loss += hyper_loss.item()
                    total_count += 1
                
                pbar.set_postfix({
                    'hyp': f'{total_hyper_loss/max(total_count,1):.4f}'
                })
        
        avg_hyper_loss = total_hyper_loss / max(total_count, 1)
        scheduler.step(avg_hyper_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        
        avg_domain_loss = total_domain_loss / max(total_count, 1)
        avg_mmd_loss = total_mmd_loss / max(total_count, 1)
        
        print(f"    Epoch {epoch+1}/{num_epochs} [LR={current_lr:.2e}]: "
              f"域={avg_domain_loss:.4f}, "
              f"MMD={avg_mmd_loss:.4f}, "
              f"超图={avg_hyper_loss:.4f}")
        
        if avg_domain_loss < best_loss:
            best_loss = avg_domain_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(mult_task_model.state_dict())
            best_model_saved = False
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print(f"    ⚠️ 早停触发: 连续{early_stop_patience}个epoch无改善，停止训练")
                if best_model_state is not None:
                    mult_task_model.load_state_dict(best_model_state)
                break
                break
    
    if best_model_state is not None:
        mult_task_model.load_state_dict(best_model_state)
        if not best_model_saved:
            best_mult_task_path = 'stage2_hypergraph_best_domain_loss_model.pth'
            torch.save(best_model_state, best_mult_task_path)
            best_model_saved = True
            print(f"  ✓ 最佳域损失模型已保存: {best_mult_task_path}")
    
    trained_models = {}
    for physical_joint in available_joints:
        single_model = DomainAdversarialModel(
            num_classes=joint_num_classes[physical_joint], 
            backbone=backbone
        )
        single_model.features = mult_task_model.features
        single_model.pool = mult_task_model.pool
        single_model.label_classifier = mult_task_model.label_classifiers[physical_joint]
        single_model.domain_classifier = mult_task_model.domain_classifier
        
        model_path = f'stage2_{physical_joint}_model.pth'
        torch.save(single_model.state_dict(), model_path)
        trained_models[physical_joint] = model_path
        print(f"  ✓ {physical_joint}模型已保存: {model_path}")
    
    del mult_task_model, single_model
    clear_gpu_memory()
    
    print(f"\n{'='*60}")
    print(f"Stage2 超图感知DANN训练完成！共 {len(trained_models)} 个模型")
    print(f"{'='*60}")
    
    return trained_models


def stage3_formula_bone_age(adapted_models: Dict[str, str],
                          rsna_train: pd.DataFrame,
                          device: torch.device,
                          num_epochs: int = 12,
                          main_model=None,
                          val_ratio: float = 0.15,
                          early_stop_patience: int = 5) -> Dict[str, str]:
    """Stage3: 语义对齐 + 公式法骨龄预测（带验证和最佳模型保存）
    
    Args:
        adapted_models: Stage2训练好的模型路径字典
        rsna_train: RSNA骨龄训练数据
        device: 训练设备
        num_epochs: 训练轮数
        main_model: 主模型（可选）
        val_ratio: 验证集比例
        early_stop_patience: 早停耐心值
    """
    
    print("\n" + "=" * 60)
    print("        Stage3: 语义对齐 + 公式法骨龄训练")
    print("=" * 60)
    
    clear_gpu_memory()
    print("✓ GPU显存已清理")
    
    from sklearn.model_selection import train_test_split
    train_data, val_data = train_test_split(
        rsna_train, 
        test_size=val_ratio, 
        random_state=42
    )
    print(f"\n数据划分: 训练集 {len(train_data)}, 验证集 {len(val_data)}")
    
    import os
    joint_model_info = {}
    for joint_name, model_path in adapted_models.items():
        if not os.path.exists(model_path):
            print(f"  跳过 {joint_name}: 模型文件不存在 {model_path}")
            continue
        
        checkpoint = torch.load(model_path, weights_only=True)
        num_classes = get_num_classes_from_checkpoint(checkpoint)
        
        joint_model_info[joint_name] = {
            'path': model_path,
            'num_classes': num_classes,
            'checkpoint': checkpoint
        }
        print(f"  ✓ 已读取关节模型信息 {joint_name} (num_classes={num_classes})")
    
    print(f"已读取 {len(joint_model_info)} 个物理关节模型信息（将分批加载到GPU）")
    
    print("\n语义映射（物理关节 → 医学关节）:")
    for medical_joint in Config.MEDICAL_JOINT_NAMES:
        physical_joints = Config.PHYSICAL_TO_MEDICAL_MAPPING.get(medical_joint, [medical_joint])
        print(f"  {medical_joint:12s} ← {', '.join(physical_joints)}")
    
    train_dataset = RSNADataset(train_data, Config.RSNA_IMG_DIR, transform=None)
    val_dataset = RSNADataset(val_data, Config.RSNA_IMG_DIR, transform=None)
    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    print(f"\n使用batch size: {batch_size}")
    
    print("\n开始训练，使用RUS-CHN公式计算骨龄...")
    
    best_val_loss = float('inf')
    best_checkpoints = {joint_name: info['checkpoint'].copy() for joint_name, info in joint_model_info.items()}
    patience_counter = 0
    
    for epoch in range(num_epochs):
        joint_models = {}
        for joint_name in joint_model_info.keys():
            info = joint_model_info[joint_name]
            ckpt = info['checkpoint']
            
            is_dann_format = 'label_classifier.weight' in ckpt or 'domain_classifier.0.weight' in ckpt
            
            if is_dann_format:
                feat_dim = ckpt['label_classifier.weight'].shape[1]
                backbone = 'resnet50' if feat_dim == 2048 else 'efficientnet_b0'
                dann_model = DomainAdversarialModel(num_classes=info['num_classes'], backbone=backbone)
                dann_model.load_state_dict(ckpt, strict=False)
                dann_model = dann_model.to(device)
                dann_model.train()
                joint_models[joint_name] = dann_model
            elif 'conv1.weight' in ckpt:
                model = models.resnet50(weights=None)
                model.fc = nn.Linear(2048, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.train()
                joint_models[joint_name] = model
            elif 'classifier.weight' in ckpt:
                model = models.efficientnet_b0(weights=None)
                model.classifier = nn.Linear(1280, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.train()
                joint_models[joint_name] = model
            else:
                model = models.efficientnet_b0(weights=None)
                model.classifier = nn.Linear(1280, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.train()
                joint_models[joint_name] = model
        
        all_params = []
        for joint_name, model in joint_models.items():
            all_params.extend(list(model.parameters()))
        
        optimizer = torch.optim.AdamW(all_params, lr=1e-5)
        criterion = nn.L1Loss()
        
        total_loss = 0
        total_count = 0
        
        pbar = tqdm(train_loader, desc=f"  Train Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            images = batch['image'].to(device)
            bone_ages = batch['bone_age'].to(device)
            genders = batch['gender'].cpu().numpy()
            
            physical_logits = {}
            for joint_name, model in joint_models.items():
                output = model(images)
                if isinstance(output, tuple):
                    logits = output[0]
                else:
                    logits = output
                physical_logits[joint_name] = logits
            
            predicted_grades = compute_medical_grades(physical_logits, device)
            batch_size_actual = images.size(0)
            gender_flags = torch.as_tensor(genders, device=device, dtype=torch.float32)
            
            pred_ages = compute_bone_age_from_grades(predicted_grades, gender_flags, device)
            
            loss = criterion(pred_ages, bone_ages)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * batch_size_actual
            total_count += batch_size_actual
            
            pbar.set_postfix({'loss': loss.item()})
        
        train_loss = total_loss / max(total_count, 1)
        
        for joint_name, model in joint_models.items():
            joint_model_info[joint_name]['checkpoint'] = model.state_dict()
        
        del joint_models, optimizer
        clear_gpu_memory()
        
        joint_models = {}
        for joint_name in joint_model_info.keys():
            info = joint_model_info[joint_name]
            ckpt = info['checkpoint']
            
            is_dann_format = 'label_classifier.weight' in ckpt or 'domain_classifier.0.weight' in ckpt
            
            if is_dann_format:
                feat_dim = ckpt['label_classifier.weight'].shape[1]
                backbone = 'resnet50' if feat_dim == 2048 else 'efficientnet_b0'
                dann_model = DomainAdversarialModel(num_classes=info['num_classes'], backbone=backbone)
                dann_model.load_state_dict(ckpt, strict=False)
                dann_model = dann_model.to(device)
                dann_model.eval()
                joint_models[joint_name] = dann_model
            elif 'conv1.weight' in ckpt:
                model = models.resnet50(weights=None)
                model.fc = nn.Linear(2048, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.eval()
                joint_models[joint_name] = model
            elif 'classifier.weight' in ckpt:
                model = models.efficientnet_b0(weights=None)
                model.classifier = nn.Linear(1280, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.eval()
                joint_models[joint_name] = model
            else:
                model = models.efficientnet_b0(weights=None)
                model.classifier = nn.Linear(1280, info['num_classes'])
                model.load_state_dict(ckpt)
                model = model.to(device)
                model.eval()
                joint_models[joint_name] = model
        
        val_loss = 0
        val_count = 0
        val_mae = 0
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                bone_ages = batch['bone_age'].to(device)
                genders = batch['gender'].cpu().numpy()
                
                physical_logits = {}
                for joint_name, model in joint_models.items():
                    output = model(images)
                    if isinstance(output, tuple):
                        logits = output[0]
                    else:
                        logits = output
                    physical_logits[joint_name] = logits
                
                predicted_grades = compute_medical_grades(physical_logits, device)
                batch_size_actual = images.size(0)
                gender_flags = torch.as_tensor(genders, device=device, dtype=torch.float32)
                
                pred_ages = compute_bone_age_from_grades(predicted_grades, gender_flags, device)
                
                loss = criterion(pred_ages, bone_ages)
                val_loss += loss.item() * batch_size_actual
                val_count += batch_size_actual
                val_mae += torch.abs(pred_ages - bone_ages).sum().item()
        
        val_loss = val_loss / max(val_count, 1)
        val_mae = val_mae / max(val_count, 1)
        
        marker = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_checkpoints = {joint_name: info['checkpoint'].copy() for joint_name, info in joint_model_info.items()}
            patience_counter = 0
            marker = " ✓"
        else:
            patience_counter += 1
        
        print(f"  Epoch {epoch+1}/{num_epochs}: train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}, val_MAE={val_mae:.2f} months{marker}")
        
        del joint_models
        clear_gpu_memory()
        
        if patience_counter >= early_stop_patience:
            print(f"  ⚠️ 早停触发: 连续{early_stop_patience}个epoch无改善，最佳val_loss={best_val_loss:.4f}")
            break
    
    for joint_name in joint_model_info.keys():
        joint_model_info[joint_name]['checkpoint'] = best_checkpoints[joint_name]
    
    trained_models = {}
    for joint_name, info in joint_model_info.items():
        model_path = f'stage3_{joint_name}_model.pth'
        torch.save(info['checkpoint'], model_path)
        trained_models[joint_name] = model_path
    
    del joint_model_info, best_checkpoints
    clear_gpu_memory()
    
    print(f"\n{'='*60}")
    print(f"Stage3训练完成！已保存 {len(trained_models)} 个关节模型")
    print(f"最佳验证损失: {best_val_loss:.4f}")
    print(f"{'='*60}")
    for model_path in trained_models.values():
        print(f"  ✓ {model_path}")
    
    return trained_models


def compute_medical_grades(physical_logits: dict, device: torch.device) -> torch.Tensor:
    """从物理关节logits计算医学关节等级"""
    return compute_medical_grades_from_physical_logits(physical_logits, device)


def compute_bone_age_from_grades(predicted_grades: torch.Tensor, gender_flags: torch.Tensor, device: torch.device) -> torch.Tensor:
    """从医学关节等级计算骨龄，返回单位: 月。"""
    del device
    return compute_bone_age_months_from_grades(predicted_grades, gender_flags)


## 📊 第三步：查看配置

In [9]:
# 打印配置
Config.print_config()

# 查看关节名称
print("\n📌 物理关节列表:")
print(Config.PHYSICAL_JOINT_NAMES)

print("\n📌 医学关节列表:")
print(Config.MEDICAL_JOINT_NAMES)

# 查看映射关系
print("\n📌 物理关节 → 医学关节 映射:")
for physical, medical in Config.PHYSICAL_TO_MEDICAL_MAPPING.items():
    print(f"  {physical:12s} → {medical}")

               配置参数
设备: cuda
源域路径: /kaggle/input/private-dataset/arthrosis
目标域路径: /kaggle/input/datasets/lihaohao111/jointsadaptation/joints_output
RSNA CSV: /kaggle/input/rsna-bone-age/boneage-training-dataset.csv
RSNA图像: /kaggle/input/datasets/lihaohao111/jointsadaptation/joints_output
YOLO权重: /kaggle/input/models/lihaohao111/bestyolo/pytorch/default/1/best (1).pt
图像尺寸: 256
D_MODEL: 512
NHEAD: 8
NUM_LAYERS: 6
Batch Size: 32
Stage1 Epochs: 10
Stage2 Epochs: 30
Stage3 Epochs: 10

📌 物理关节列表:
['Radius', 'Ulna', 'MCPFirst', 'MCP', 'PIP', 'DIP', 'MIP', 'DIPFirst', 'PIPFirst']

📌 医学关节列表:
['Radius', 'Ulna', 'MCPFirst', 'MCPThird', 'MCPFifth', 'PIPFirst', 'PIPThird', 'PIPFifth', 'MIPThird', 'MIPFifth', 'DIPFirst', 'DIPThird', 'DIPFifth']

📌 物理关节 → 医学关节 映射:
  Radius       → ['Radius']
  Ulna         → ['Ulna']
  MCPFirst     → ['MCPFirst']
  MCP          → ['MCPThird', 'MCPFifth']
  PIP          → ['PIPThird', 'PIPFifth']
  DIP          → ['DIPThird', 'DIPFifth']
  MIP          → ['MIPThird', '

## 📁 第四步：准备数据

In [10]:
# 数据准备
print("=" * 60)
print("数据准备")
print("=" * 60)

# 源域数据 (Arthritis)
if os.path.exists(Config.SOURCE_DATA_ROOT):
    source_train, source_val, source_test = split_arthrosis_dataset(
        Config.SOURCE_DATA_ROOT,
        Config.TRAIN_RATIO,
        Config.VAL_RATIO,
        Config.TEST_RATIO
    )
    print(f"源域: 训练={len(source_train)}, 验证={len(source_val)}, 测试={len(source_test)}")
else:
    print(f"⚠️  源域数据不存在: {Config.SOURCE_DATA_ROOT}")
    source_train = pd.DataFrame()
    source_test = pd.DataFrame()

# 目标域数据
if os.path.exists(Config.TARGET_DATA_ROOT):
    target_train, target_val, target_test = split_arthrosis_dataset(
        Config.TARGET_DATA_ROOT,
        0.667, 0.333, 0.333
    )
    print(f"目标域: 训练={len(target_train)}, 验证={len(target_val)}, 测试={len(target_test)}")
else:
    print(f"⚠️  目标域数据不存在: {Config.TARGET_DATA_ROOT}")
    target_train = pd.DataFrame()
    target_test = pd.DataFrame()

# RSNA数据
if os.path.exists(Config.RSNA_CSV_PATH):
    rsna_train, rsna_val, rsna_test = prepare_rsna_dataset(
        Config.RSNA_CSV_PATH,
        Config.RSNA_IMG_DIR,
        Config.RSNA_SAMPLES
    )
    print(f"RSNA: 训练={len(rsna_train)}, 验证={len(rsna_val)}, 测试={len(rsna_test)}")
else:
    print(f"⚠️  RSNA数据不存在: {Config.RSNA_CSV_PATH}")
    rsna_train = pd.DataFrame()
    rsna_test = pd.DataFrame()

数据准备
  发现关节: PIPFirst
  发现关节: MCP
  发现关节: MCPFirst
  发现关节: MIP
  发现关节: Radius
  发现关节: PIP
  发现关节: Ulna
  发现关节: DIPFirst
  发现关节: DIP
  使用关节列表: ['PIPFirst', 'MCP', 'MCPFirst', 'MIP', 'Radius', 'PIP', 'Ulna', 'DIPFirst', 'DIP']
  总样本数: 8210
源域: 训练=5747, 验证=1231, 测试=1232
  发现关节: PIPFirst
  发现关节: MCP
  发现关节: MCPFirst
  发现关节: MIP
  发现关节: Radius
  发现关节: PIP
  发现关节: Ulna
  发现关节: DIPFirst
  发现关节: DIP
  使用关节列表: ['PIPFirst', 'MCP', 'MCPFirst', 'MIP', 'Radius', 'PIP', 'Ulna', 'DIPFirst', 'DIP']
  总样本数: 105
目标域: 训练=35, 验证=35, 测试=35
RSNA: 训练=700, 验证=150, 测试=150


## 🏗️ 第五步：初始化模型

In [11]:
# 创建模型
print("\n" + "=" * 60)
print("模型初始化")
print("=" * 60)

model = CNNTransformerBoneAge(
    num_classes_dict=Config.MEDICAL_JOINT_NUM_CLASSES,
    d_model=Config.D_MODEL,
    nhead=Config.NHEAD,
    num_layers=Config.NUM_LAYERS,
    dropout=Config.DROPOUT
).to(Config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"模型参数量: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print(f"设备: {Config.DEVICE}")


模型初始化
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 226MB/s]


模型参数量: 43,686,885
可训练参数: 43,686,885
设备: cuda


 86%|████████▋ | 84.5M/97.8M [00:00<00:00, 182MB/s]

100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s]

模型参数量: 43,686,885
可训练参数: 43,686,885
设备: cuda


## 🔄 数据增强配置

In [12]:
# 数据增强
train_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✅ 数据增强配置完成")

✅ 数据增强配置完成


In [13]:
"""
主入口模块

二阶段递进式骨龄预测训练脚本

训练流程:
1. Stage1: 源域特征学习 (Arthritis关节分类)
2. Stage2: 域自适应对齐 (DANN + GRL + MMD)
"""

import os
import json
import pandas as pd
import torch
from torchvision import transforms


def main():
    """主训练流程"""
    
    Config.print_config()
    
    print("\n" + "=" * 60)
    print("          二阶段递进式骨龄预测训练")
    print("=" * 60)
    
    # ========== 数据准备 ==========
    print("\n[数据准备]")
    
    if os.path.exists(Config.SOURCE_DATA_ROOT):
        source_train, source_val, source_test = split_arthrosis_dataset(
            Config.SOURCE_DATA_ROOT,
            Config.TRAIN_RATIO,
            Config.VAL_RATIO,
            Config.TEST_RATIO
        )
        print(f"  源域: 训练={len(source_train)}, 验证={len(source_val)}, 测试={len(source_test)}")
    else:
        print(f"  警告: 源域数据不存在 {Config.SOURCE_DATA_ROOT}")
        source_train = pd.DataFrame()
        source_test = pd.DataFrame()
    
    if os.path.exists(Config.TARGET_DATA_ROOT):
        target_train, target_val, target_test = split_arthrosis_dataset(
            Config.TARGET_DATA_ROOT,
            0.667, 0.333, 0.333
        )
        print(f"  目标域: 训练={len(target_train)}, 验证={len(target_val)}, 测试={len(target_test)}")
    else:
        print(f"  警告: 目标域数据不存在 {Config.TARGET_DATA_ROOT}")
        target_train = pd.DataFrame()
        target_test = pd.DataFrame()
    
    if os.path.exists(Config.RSNA_CSV_PATH):
        rsna_train, rsna_val, rsna_test = prepare_rsna_dataset(
            Config.RSNA_CSV_PATH,
            Config.RSNA_IMG_DIR,
            Config.RSNA_SAMPLES
        )
        print(f"  RSNA: 训练={len(rsna_train)}, 验证={len(rsna_val)}, 测试={len(rsna_test)}")
    else:
        print(f"  警告: RSNA数据不存在 {Config.RSNA_CSV_PATH}")
        rsna_train = pd.DataFrame()
        rsna_test = pd.DataFrame()
    
    # ========== 模型初始化 ==========
    print("\n[模型初始化]")
    
    model = CNNTransformerBoneAge(
        num_classes_dict=Config.MEDICAL_JOINT_NUM_CLASSES,
        d_model=Config.D_MODEL,
        nhead=Config.NHEAD,
        num_layers=Config.NUM_LAYERS,
        dropout=Config.DROPOUT,
        backbone='resnet50'
    ).to(Config.DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  模型参数总数: {total_params:,}")
    
    # ========== 数据增强 ==========
    train_transform = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # ========== Stage1: 源域特征学习 ==========
    if len(source_train) > 0:
        stage1_models = stage1_source_feature_learning(
            source_train,
            Config.DEVICE,
            Config.EPOCHS_STAGE1
        )
        print(f"\nStage1训练完成！共 {len(stage1_models)} 个关节模型")
    else:
        print("\n  跳过Stage1: 源域训练数据为空")
        stage1_models = {}
    
    # ========== Stage2: DANN域自适应泛化训练 ==========
    if len(stage1_models) > 0 and len(target_train) > 0:
        stage2_models = stage2_domain_adaptation(
            stage1_models,
            target_train,
            Config.DEVICE,
            num_epochs=Config.EPOCHS_STAGE2,
            batch_size=Config.BATCH_SIZE,
            train_transform=train_transform,
            source_train=source_train
        )
        print(f"\nStage2 DANN域自适应完成！共 {len(stage2_models)} 个关节模型")
    else:
        print("\n  跳过Stage2: Stage1模型或目标域数据为空")
        stage2_models = stage1_models
    

    
    # ========== 测试评估 ==========
    print("\n" + "=" * 60)
    print("               测试评估")
    print("=" * 60)
    
    torch.save(model.state_dict(), 'final_bone_age_model.pth')
    print("\n最终模型已保存: final_bone_age_model.pth")
    
    print("\n" + "=" * 60)
    print("               完整测试评估")
    print("=" * 60)
    
    test_transform = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    joint_results = None
    target_joint_results = None
    mae_results = None
    domain_results = None
    
    if len(source_test) > 0:
        print("\n[1/3] 测试源域（Arthritis）关节分类...")
        source_test_dataset = ArthrosisDataset(source_test, transform=test_transform)
        source_test_loader = torch.utils.data.DataLoader(
            source_test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0
        )
        if len(stage1_models) > 0:
            joint_results = evaluate_joint_classification(
                stage1_models, source_test_loader, Config.DEVICE, save_dir='plots/source_domain'
            )
        elif len(stage2_models) > 0:
            joint_results = evaluate_joint_classification(
                stage2_models, source_test_loader, Config.DEVICE, save_dir='plots/source_domain'
            )
        else:
            joint_results = None
    else:
        print("\n[1/3] 跳过源域测试：测试集为空")
        joint_results = None
    
    if len(target_test) > 0:
        print("\n[2/3] 测试目标域（Arthritis）关节分类...")
        target_test_dataset = ArthrosisDataset(target_test, transform=test_transform)
        target_test_loader = torch.utils.data.DataLoader(
            target_test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0
        )
        if len(stage2_models) > 0:
            target_joint_results = evaluate_joint_classification(
                stage2_models, target_test_loader, Config.DEVICE, save_dir='plots/target_domain'
            )
        elif len(stage1_models) > 0:
            target_joint_results = evaluate_joint_classification(
                stage1_models, target_test_loader, Config.DEVICE, save_dir='plots/target_domain'
            )
        else:
            target_joint_results = None
    else:
        print("\n[2/3] 跳过目标域测试：测试集为空")
        target_joint_results = None
    
    if len(source_test) > 0 and len(target_test) > 0:
        print("\n[3/3] 域自适应性能评估...")
        source_test_dataset = ArthrosisDataset(source_test, transform=test_transform)
        target_test_dataset = ArthrosisDataset(target_test, transform=test_transform)
        source_test_loader = torch.utils.data.DataLoader(
            source_test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0
        )
        target_test_loader = torch.utils.data.DataLoader(
            target_test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0
        )
        if len(stage2_models) > 0:
            domain_results = evaluate_domain_loss(
                stage2_models, source_test_loader, target_test_loader, Config.DEVICE, save_dir='plots/domain_adaptation'
            )
        elif len(stage1_models) > 0:
            domain_results = evaluate_domain_loss(
                stage1_models, source_test_loader, target_test_loader, Config.DEVICE, save_dir='plots/domain_adaptation'
            )
        else:
            domain_results = {}
    else:
        print("\n[3/3] 跳过域自适应测试：数据集不足")
        domain_results = {}
    
    mae_results = None
    if len(rsna_test) > 0:
        print("\n[4/4] 测试RSNA骨龄预测MAE...")
        rsna_test_dataset = RSNADataset(rsna_test, Config.RSNA_IMG_DIR, transform=test_transform)
        rsna_test_loader = torch.utils.data.DataLoader(
            rsna_test_dataset, batch_size=16, shuffle=False, num_workers=0
        )
        if len(stage2_models) > 0:
            mae_results = evaluate_bone_age_mae(
                stage2_models, rsna_test_loader, Config.DEVICE, save_dir='plots/rsna_bone_age'
            )
        elif len(stage1_models) > 0:
            mae_results = evaluate_bone_age_mae(
                stage1_models, rsna_test_loader, Config.DEVICE, save_dir='plots/rsna_bone_age'
            )
        else:
            mae_results = None
    else:
        print("\n[4/4] 跳过RSNA测试：测试集为空")
    
    print("\n" + "=" * 60)
    print("               测试完成总结")
    print("=" * 60)
    
    summary = {
        'stage1_source_joint_acc': joint_results if joint_results else None,
        'stage2_target_joint_acc': target_joint_results if target_joint_results else None,
        'stage3_rsna_mae': mae_results if mae_results else None,
        'domain_adaptation': domain_results if domain_results else None,
    }
    33
    def convert_to_serializable(obj):
        if isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [convert_to_serializable(item) for item in obj]
        elif isinstance(obj, (int, float)):
            return float(obj) if obj is not None else None
        else:
            return obj
    
    with open('test_summary.json', 'w', encoding='utf-8') as f:
        json.dump(convert_to_serializable(summary), f, ensure_ascii=False, indent=2)
    print("✓ 测试摘要已保存: test_summary.json")
    
    print("\n" + "=" * 60)
    print("               训练流程全部完成！")
    print("=" * 60)
    print("\n生成的文件：")
    print("  模型文件:")
    print("    - stage1_source_model.pth      (Stage1源域特征模型)")
    print("    - stage2_domain_model.pth     (Stage2域自适应模型)")
    print("    - final_bone_age_model.pth    (最终完整模型)")
    print("\n  评估报告:")
    print("    - test_summary.json           (测试结果摘要)")
    print("\n  ROC曲线和可视化图表:")
    if len(source_test) > 0:
        print("    - plots/source_domain/           (源域关节分类ROC曲线)")
    if len(target_test) > 0:
        print("    - plots/target_domain/           (目标域关节分类ROC曲线)")
    if len(source_test) > 0 and len(target_test) > 0:
        print("    - plots/domain_adaptation/        (域自适应分析图)")
    if len(rsna_test) > 0:
        print("    - plots/rsna_bone_age/            (骨龄预测分析图)")
    print("\n训练完成！")


if __name__ == '__main__':
    RUN_TEST = False
    
    if RUN_TEST:
        print("运行ROC曲线测试...")
        from .visualization import test_roc_plotting
        test_roc_plotting()
    else:
        print("运行训练流程...")
        #main()


运行训练流程...


In [14]:
"""
评估模块

包含:
- plot_roc_curve: 单类别ROC曲线绘制
- plot_multiclass_roc: 多类别ROC曲线绘制
- evaluate_joint_classification: 关节分类评估（支持单关节模型）
- evaluate_domain_loss: 域损失评估（支持DANN模型）
- evaluate_bone_age_mae: 骨龄MAE评估（支持多关节融合）
"""

import os
import inspect
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc, accuracy_score, mean_absolute_error
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from torchvision import models as torchvision_models
from torchvision import transforms
from PIL import Image
import cv2






PROJECT_LABEL_ALIASES = {
    'Radius': 'Radius',
    'Ulna': 'Ulna',
    'MCPFirst': 'MCPFirst',
    'MCPThird': 'MCP',
    'MCPFifth': 'MCP',
    'MCP': 'MCP',
    'PIPFirst': 'PIPFirst',
    'PIPThird': 'PIP',
    'PIPFifth': 'PIP',
    'PIP': 'PIP',
    'DIPFirst': 'DIPFirst',
    'DIPThird': 'DIP',
    'DIPFifth': 'DIP',
    'DIP': 'DIP',
    'MIPThird': 'MIP',
    'MIPFifth': 'MIP',
    'MIP': 'MIP',
}

THUMB_JOINT_ORDER = ['DIPFirst', 'PIPFirst', 'MCPFirst']
LONG_FINGER_JOINT_ORDER = ['DIP', 'MIP', 'PIP', 'MCP']
BACKEND_SIGNAL_LABELS = {
    'Radius',
    'Ulna',
    'MCPFirst',
    'MCP',
    'ProximalPhalanx',
    'MiddlePhalanx',
    'DistalPhalanx',
}
BACKEND_DETECT_TO_MODEL = {
    'DIPFirst': 'DIPFirst',
    'DIPThird': 'DIP',
    'DIPFifth': 'DIP',
    'PIPFirst': 'PIPFirst',
    'PIPThird': 'PIP',
    'PIPFifth': 'PIP',
    'MCPFirst': 'MCPFirst',
    'MCPSecond': 'MCP',
    'MCPThird': 'MCP',
    'MCPFourth': 'MCP',
    'MCPFifth': 'MCP',
    'MIPThird': 'MIP',
    'MIPFifth': 'MIP',
    'Radius': 'Radius',
    'Ulna': 'Ulna',
}
BACKEND_PREFIX_RENAME_CONFIG = {
    'MCP': {
        'source_labels': {'MCPFirst', 'MCP'},
        'joint_names': ['MCPFirst', 'MCPSecond', 'MCPThird', 'MCPFourth', 'MCPFifth'],
        'expected_count': 5,
    },
    'PIP': {
        'source_labels': {'ProximalPhalanx'},
        'joint_names': ['PIPFirst', 'PIPSecond', 'PIPThird', 'PIPFourth', 'PIPFifth'],
        'expected_count': 5,
    },
    'MIP': {
        'source_labels': {'MiddlePhalanx'},
        'joint_names': ['MIPSecond', 'MIPThird', 'MIPFourth', 'MIPFifth'],
        'expected_count': 4,
    },
    'DIP': {
        'source_labels': {'DistalPhalanx'},
        'joint_names': ['DIPFirst', 'DIPSecond', 'DIPThird', 'DIPFourth', 'DIPFifth'],
        'expected_count': 5,
    },
}


def _extract_class_logits(model_output):
    """兼容普通分类模型和 DANN 模型输出。"""
    if isinstance(model_output, tuple):
        return model_output[0]
    return model_output


def _extract_backbone_features(model: nn.Module, images: torch.Tensor) -> torch.Tensor:
    """兼容不同模型结构的特征提取。"""
    if hasattr(model, 'extract_features') and callable(model.extract_features):
        return model.extract_features(images)

    if hasattr(model, 'features') and hasattr(model, 'avgpool'):
        feats = model.features(images)
        return model.avgpool(feats).flatten(1)

    if all(hasattr(model, attr) for attr in ['conv1', 'bn1', 'relu', 'maxpool', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool']):
        feats = model.conv1(images)
        feats = model.bn1(feats)
        feats = model.relu(feats)
        feats = model.maxpool(feats)
        feats = model.layer1(feats)
        feats = model.layer2(feats)
        feats = model.layer3(feats)
        feats = model.layer4(feats)
        return model.avgpool(feats).flatten(1)

    raise TypeError(f"不支持的模型特征提取类型: {type(model).__name__}")


def _normalize_project_joint_label(label: str) -> Optional[str]:
    if not label:
        return None
    normalized = PROJECT_LABEL_ALIASES.get(label)
    if normalized in Config.PHYSICAL_JOINT_NAMES:
        return normalized
    return None


def _cluster_regions_by_x(regions: List[Dict]) -> List[List[Dict]]:
    if not regions:
        return []

    sorted_regions = sorted(regions, key=lambda region: region['centroid'][0])
    x_values = [region['centroid'][0] for region in sorted_regions]
    spread = max(x_values) - min(x_values)
    gap_threshold = max(18.0, spread / 10.0)

    clusters = [[sorted_regions[0]]]
    current_mean_x = float(sorted_regions[0]['centroid'][0])

    for region in sorted_regions[1:]:
        x = float(region['centroid'][0])
        if abs(x - current_mean_x) <= gap_threshold:
            clusters[-1].append(region)
            current_mean_x = float(np.mean([item['centroid'][0] for item in clusters[-1]]))
        else:
            clusters.append([region])
            current_mean_x = x

    clusters.sort(key=lambda cluster: float(np.mean([item['centroid'][0] for item in cluster])))
    return clusters


def _region_center_x(region: Dict) -> float:
    return float(region.get('centroid', (0.0, 0.0))[0])


def _region_center_y(region: Dict) -> float:
    return float(region.get('centroid', (0.0, 0.0))[1])


def _region_confidence(region: Dict) -> float:
    return float(region.get('confidence', 0.0))


def _region_bbox(region: Dict) -> Tuple[float, float, float, float]:
    bbox = region.get('bbox_coords', (0.0, 0.0, 0.0, 0.0))
    if len(bbox) != 4:
        return 0.0, 0.0, 0.0, 0.0
    return tuple(float(v) for v in bbox)


def _bbox_iou(
    box_a: Tuple[float, float, float, float],
    box_b: Tuple[float, float, float, float],
) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    if inter_area <= 0:
        return 0.0

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter_area
    if denom <= 0:
        return 0.0
    return inter_area / denom


def _dedupe_regions_by_overlap(
    regions: List[Dict],
    expected_count: int,
    iou_threshold: float = 0.35,
) -> List[Dict]:
    if not regions:
        return []

    ordered_by_conf = sorted(
        regions,
        key=lambda item: (
            _region_confidence(item),
            _region_bbox(item)[2] - _region_bbox(item)[0],
        ),
        reverse=True,
    )

    kept = []
    for region in ordered_by_conf:
        region_box = _region_bbox(region)
        if any(_bbox_iou(region_box, _region_bbox(existing)) >= iou_threshold for existing in kept):
            continue
        kept.append(region)

    return kept[:expected_count]


def _order_regions_by_hand_side(regions: List[Dict], hand_side: str) -> List[Dict]:
    ordered = sorted(regions, key=_region_center_x)
    if hand_side == 'left':
        ordered.reverse()
    return ordered


def _resolve_hand_side(
    radius_x: Optional[float],
    ulna_x: Optional[float],
    fallback: str = 'unknown',
) -> str:
    if radius_x is None or ulna_x is None:
        return fallback if fallback in {'left', 'right'} else 'unknown'
    return 'left' if float(radius_x) > float(ulna_x) else 'right'


def _resolve_hand_side_from_regions(
    regions: List[Dict],
    fallback: str = 'unknown',
) -> str:
    radius_regions = [region for region in regions if region.get('label') == 'Radius']
    ulna_regions = [region for region in regions if region.get('label') == 'Ulna']
    if radius_regions and ulna_regions:
        radius_region = max(radius_regions, key=_region_confidence)
        ulna_region = max(ulna_regions, key=_region_confidence)
        return _resolve_hand_side(
            _region_center_x(radius_region),
            _region_center_x(ulna_region),
            fallback=fallback,
        )
    return fallback if fallback in {'left', 'right'} else 'unknown'


def _assign_vertical_joint_names(regions: List[Dict], joint_names: List[str]) -> None:
    ordered_regions = sorted(regions, key=lambda region: region['centroid'][1])[:len(joint_names)]
    for region, joint_name in zip(ordered_regions, joint_names):
        region['project_label'] = joint_name
        region['model_joint'] = joint_name
        region['naming_source'] = 'spatial_heuristic'


def _try_backend_main_naming(regions: List[Dict], hand_side: str) -> Optional[List[Dict]]:
    available_labels = {region.get('label') for region in regions}
    if not (available_labels & BACKEND_SIGNAL_LABELS):
        return None

    renamed_regions = []
    for region in regions:
        copied = dict(region)
        copied['project_label'] = None
        copied['model_joint'] = None
        copied['naming_source'] = 'unassigned'
        renamed_regions.append(copied)

    for wrist_label in ['Radius', 'Ulna']:
        wrist_regions = [region for region in renamed_regions if region.get('label') == wrist_label]
        if not wrist_regions:
            continue
        wrist_region = max(wrist_regions, key=_region_confidence)
        wrist_region['project_label'] = wrist_label
        wrist_region['model_joint'] = BACKEND_DETECT_TO_MODEL.get(wrist_label)
        wrist_region['naming_source'] = 'backend_main'

    for _, config in BACKEND_PREFIX_RENAME_CONFIG.items():
        prefix_regions = [
            region for region in renamed_regions if region.get('label') in config['source_labels']
        ]
        if not prefix_regions:
            continue

        deduped_regions = _dedupe_regions_by_overlap(prefix_regions, config['expected_count'])
        ordered_regions = _order_regions_by_hand_side(deduped_regions, hand_side)

        for joint_name, region in zip(config['joint_names'], ordered_regions[:len(config['joint_names'])]):
            region['project_label'] = joint_name
            region['model_joint'] = BACKEND_DETECT_TO_MODEL.get(joint_name)
            region['naming_source'] = 'backend_main'

    return renamed_regions


def _rename_dpv3_regions_backend_style(
    regions: List[Dict],
    hand_side_hint: str = 'unknown',
) -> Tuple[Dict[str, Dict], List[Dict], str]:
    hand_side = _resolve_hand_side_from_regions(regions, hand_side_hint)
    joints: Dict[str, Dict] = {}
    ordered_joints: List[Dict] = []

    for wrist_label in ['Radius', 'Ulna']:
        wrist_regions = [region for region in regions if region.get('label') == wrist_label]
        if not wrist_regions:
            continue
        wrist_region = max(wrist_regions, key=_region_confidence)
        payload = dict(wrist_region)
        payload['project_label'] = wrist_label
        payload['model_joint'] = BACKEND_DETECT_TO_MODEL.get(wrist_label)
        payload['naming_source'] = 'backend_exact'
        joints[wrist_label] = payload
        ordered_joints.append(payload)

    for _, config in BACKEND_PREFIX_RENAME_CONFIG.items():
        prefix_regions = [
            region for region in regions if region.get('label') in config['source_labels']
        ]
        if not prefix_regions:
            continue

        deduped_regions = _dedupe_regions_by_overlap(prefix_regions, config['expected_count'])
        ordered_regions = _order_regions_by_hand_side(deduped_regions, hand_side)

        for joint_name, region in zip(config['joint_names'], ordered_regions[: len(config['joint_names'])]):
            payload = dict(region)
            payload['project_label'] = joint_name
            payload['model_joint'] = BACKEND_DETECT_TO_MODEL.get(joint_name)
            payload['naming_source'] = 'backend_exact'
            joints[joint_name] = payload
            ordered_joints.append(payload)

    return joints, ordered_joints, hand_side


def _infer_project_named_regions(regions: List[Dict], hand_side: str) -> List[Dict]:
    backend_joints, ordered_backend_joints, resolved_hand_side = _rename_dpv3_regions_backend_style(
        regions,
        hand_side_hint=hand_side,
    )
    if backend_joints:
        return ordered_backend_joints

    hand_side = resolved_hand_side
    backend_named_regions = _try_backend_main_naming(regions, hand_side)
    if backend_named_regions is not None:
        return backend_named_regions

    renamed_regions = []
    for region in regions:
        copied = dict(region)
        copied['project_label'] = _normalize_project_joint_label(region.get('label'))
        copied['model_joint'] = copied['project_label']
        copied['naming_source'] = 'existing_label' if copied['project_label'] else 'unassigned'
        renamed_regions.append(copied)

    assigned_unique = {
        region['project_label']
        for region in renamed_regions
        if region['project_label'] in ('Radius', 'Ulna', 'MCPFirst', 'PIPFirst', 'DIPFirst')
    }

    unassigned = [region for region in renamed_regions if region['project_label'] is None]

    if 'Radius' not in assigned_unique or 'Ulna' not in assigned_unique:
        forearm_candidates = sorted(
            unassigned,
            key=lambda region: (region['centroid'][1], region.get('area', 0)),
            reverse=True,
        )[:2]
        if len(forearm_candidates) == 2:
            left_region, right_region = sorted(forearm_candidates, key=lambda region: region['centroid'][0])
            if hand_side == 'right':
                ordered_assignment = [('Radius', left_region), ('Ulna', right_region)]
            else:
                ordered_assignment = [('Ulna', left_region), ('Radius', right_region)]

            for joint_name, region in ordered_assignment:
                if joint_name not in assigned_unique and region['project_label'] is None:
                    region['project_label'] = joint_name
                    region['model_joint'] = joint_name
                    region['naming_source'] = 'spatial_heuristic'
                    assigned_unique.add(joint_name)

    unassigned = [region for region in renamed_regions if region['project_label'] is None]
    clusters = _cluster_regions_by_x(unassigned)

    if clusters:
        if hand_side == 'right':
            thumb_cluster = clusters[0]
            fifth_cluster = clusters[-1]
        elif hand_side == 'left':
            thumb_cluster = clusters[-1]
            fifth_cluster = clusters[0]
        else:
            thumb_cluster = clusters[0] if len(clusters[0]) <= len(clusters[-1]) else clusters[-1]
            fifth_cluster = clusters[-1] if thumb_cluster is clusters[0] else clusters[0]

        remaining_clusters = [cluster for cluster in clusters if cluster is not thumb_cluster and cluster is not fifth_cluster]
        middle_cluster = remaining_clusters[len(remaining_clusters) // 2] if remaining_clusters else None

        if 'MCPFirst' not in assigned_unique and 'PIPFirst' not in assigned_unique and 'DIPFirst' not in assigned_unique:
            _assign_vertical_joint_names(thumb_cluster, THUMB_JOINT_ORDER)

        if middle_cluster is not None:
            _assign_vertical_joint_names(middle_cluster, LONG_FINGER_JOINT_ORDER)

        if fifth_cluster is not thumb_cluster:
            _assign_vertical_joint_names(fifth_cluster, LONG_FINGER_JOINT_ORDER)

    return renamed_regions


def _get_joint_eval_transform(test_loader: DataLoader):
    dataset = getattr(test_loader, 'dataset', None)
    dataset_transform = getattr(dataset, 'transform', None)
    if callable(dataset_transform):
        return dataset_transform
    return transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def _crop_region_from_image(image_bgr: np.ndarray, bbox_coords: Tuple[int, int, int, int]) -> Optional[np.ndarray]:
    x1, y1, x2, y2 = bbox_coords
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(image_bgr.shape[1], int(x2)), min(image_bgr.shape[0], int(y2))
    if x2 <= x1 or y2 <= y1:
        return None
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return crop


def _prepare_crop_tensor(crop_bgr: np.ndarray, transform, device: torch.device) -> torch.Tensor:
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_pil = Image.fromarray(crop_rgb)
    crop_tensor = transform(crop_pil)
    if crop_tensor.dim() == 3:
        crop_tensor = crop_tensor.unsqueeze(0)
    return crop_tensor.to(device)


def _load_image_bgr(image_path: str) -> Optional[np.ndarray]:
    if not image_path or not os.path.exists(image_path):
        return None

    image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_bgr is not None:
        return image_bgr

    try:
        image_rgb = np.array(Image.open(image_path).convert('RGB'))
        return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    except Exception:
        return None


def _resolve_image_path(image_path: Optional[str], dataset=None) -> Optional[str]:
    if not image_path:
        return None

    candidate_paths = []

    def _add_candidate(path_value: Optional[str]) -> None:
        if not path_value:
            return
        normalized = os.path.normpath(path_value)
        candidate_paths.append(normalized)
        candidate_paths.append(os.path.abspath(normalized))

    _add_candidate(image_path)

    dataset_img_dir = getattr(dataset, 'img_dir', None)
    image_basename = os.path.basename(image_path)

    if dataset_img_dir:
        _add_candidate(os.path.join(dataset_img_dir, image_path))
        _add_candidate(os.path.join(dataset_img_dir, image_basename))

    config_img_dir = getattr(Config, 'RSNA_IMG_DIR', '')
    if config_img_dir:
        _add_candidate(os.path.join(config_img_dir, image_path))
        _add_candidate(os.path.join(config_img_dir, image_basename))

    checked = set()
    for candidate in candidate_paths:
        if not candidate or candidate in checked:
            continue
        checked.add(candidate)
        if os.path.exists(candidate):
            return candidate

    return None


def _supports_kwarg(func, kwarg_name: str) -> bool:
    try:
        return kwarg_name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


def _load_yolo_joint_model(model_path: Optional[str] = None):
    resolved_path = model_path or os.environ.get('YOLO_JOINT_MODEL_PATH') or getattr(Config, 'YOLO_JOINT_MODEL_PATH', '')
    print(f"  YOLO权重候选路径: {resolved_path or '未设置'}")
    if not resolved_path:
        print("  ⚠️ 未配置YOLO关节检测权重路径，将仅使用DPV3启发式分割")
        return None

    if not os.path.exists(resolved_path):
        print(f"  ⚠️ YOLO关节检测权重不存在: {resolved_path}")
        return None

    try:
        from ultralytics import YOLO
    except ImportError as e:
        print(f"  ⚠️ 无法导入ultralytics，跳过YOLO检测: {e}")
        return None

    try:
        yolo_model = YOLO(resolved_path)
        print(f"  ✓ YOLO关节检测模型已加载: {resolved_path}")
        return yolo_model
    except Exception as e:
        print(f"  ⚠️ YOLO关节检测模型加载失败: {e}")
        return None


def plot_roc_curve(y_true: np.ndarray, y_scores: np.ndarray, 
                  title: str, save_path: str) -> float:
    """单类别ROC曲线绘制"""
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return roc_auc


def plot_multiclass_roc(y_true: np.ndarray, y_probs: np.ndarray,
                       class_names: List[str], save_path: str,
                       title: str = "ROC Curves") -> Tuple[float, Dict]:
    """多类别ROC曲线绘制"""
    n_classes = y_probs.shape[1]
    
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        if len(np.unique(y_true_bin[:, i])) > 1:
            fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
    
    if not fpr:
        print(f"  警告: {title} - 所有类别样本数不足，无法计算ROC曲线")
        return 0.0, {}
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes) if i in fpr]))
    mean_tpr = np.zeros_like(all_fpr)
    
    for i in range(n_classes):
        if i in fpr:
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    
    mean_tpr /= len([i for i in range(n_classes) if i in fpr])
    mean_auc = auc(all_fpr, mean_tpr)
    
    plt.figure(figsize=(10, 8))
    
    for i in range(n_classes):
        if i in roc_auc:
            plt.plot(fpr[i], tpr[i], lw=2,
                    label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')
    
    plt.plot(all_fpr, mean_tpr, color='navy', lw=2, linestyle='--',
            label=f'Mean ROC (AUC = {mean_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return mean_auc, roc_auc


def load_single_joint_model(model_path: str, num_classes: int, device: torch.device) -> nn.Module:
    """加载单个关节模型（ResNet50）
    
    Args:
        model_path: 模型权重文件路径
        num_classes: 类别数量
        device: 设备
    Returns:
        加载好的模型
    """
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device, weights_only=True)

        is_dann_format = 'label_classifier.weight' in checkpoint or 'domain_classifier.0.weight' in checkpoint
        if is_dann_format:
            feat_dim = checkpoint['label_classifier.weight'].shape[1]
            backbone = 'resnet50' if feat_dim == 2048 else 'efficientnet_b0'
            model = DomainAdversarialModel(num_classes=num_classes, backbone=backbone)
        elif 'classifier.weight' in checkpoint and 'fc.weight' not in checkpoint and 'fc.1.weight' not in checkpoint:
            model = torchvision_models.efficientnet_b0(weights=None)
            model.classifier = nn.Linear(1280, num_classes)
        else:
            model = torchvision_models.resnet50(weights=None)
            model.fc = nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(2048, num_classes)
            )

        missing_keys, unexpected_keys = model.load_state_dict(checkpoint, strict=False)
        if missing_keys:
            print(f"    警告: 缺少的键: {missing_keys[:3]}...")
        if unexpected_keys:
            print(f"    警告: 多余的键: {unexpected_keys[:3]}...")
        print(f"    已加载模型: {model_path}")
    else:
        model = torchvision_models.resnet50(weights=None)
        model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(2048, num_classes)
        )
        print(f"    警告: 模型文件不存在 {model_path}")
    
    model = model.to(device)
    model.eval()
    return model


def get_num_classes_from_model(model_path: str, device: torch.device) -> int:
    """从模型文件获取类别数量"""
    if not os.path.exists(model_path):
        return 8
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    
    weight_keys = ['fc.weight', 'fc.1.weight', 'classifier.weight', 'label_classifier.weight']
    for key in weight_keys:
        if key in checkpoint:
            return checkpoint[key].shape[0]
    
    if 'num_classes' in checkpoint:
        return checkpoint['num_classes']
    
    return 8


def evaluate_joint_classification(
    trained_models: Dict[str, str],
    test_loader: DataLoader,
    device: torch.device,
    save_dir: str = 'plots'
) -> Dict:
    """评估关节分类准确率（支持分立单关节模型）
    
    Args:
        trained_models: 训练好的模型路径字典，格式: {physical_joint_name: model_path}
        test_loader: 测试数据加载器
        device: 设备
        save_dir: 保存目录
    Returns:
        各关节分类准确率字典
    """
    os.makedirs(save_dir, exist_ok=True)
    
    print("\n" + "=" * 60)
    print("              关节分类准确率测试")
    print("=" * 60)
    
    results = {}
    auc_results = {}
    
    for physical_joint in Config.PHYSICAL_JOINT_NAMES:
        if physical_joint not in trained_models:
            continue
        
        model_path = trained_models[physical_joint]
        if not os.path.exists(model_path):
            print(f"\n跳过 {physical_joint}: 模型文件不存在")
            continue
        
        num_classes = get_num_classes_from_model(model_path, device)
        model = load_single_joint_model(model_path, num_classes, device)
        
        joint_data = []
        joint_labels = []
        joint_probs = []
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"评估 {physical_joint}"):
                images = batch['image'].to(device)
                grades = batch['grade'].to(device)
                
                if 'joint' in batch:
                    batch_joints = batch['joint']
                    if isinstance(batch_joints, torch.Tensor):
                        if batch_joints.dim() > 0:
                            batch_joints = batch_joints.cpu().tolist()
                        else:
                            batch_joints = [batch_joints.item()]
                    elif isinstance(batch_joints, list):
                        batch_joints = [str(j) for j in batch_joints]
                    else:
                        batch_joints = [str(batch_joints)]
                else:
                    batch_joints = [None] * images.size(0)
                
                valid_indices = [i for i, pj in enumerate(batch_joints) 
                               if pj == physical_joint or pj is None]
                
                if valid_indices:
                    batch_images = images[valid_indices]
                    batch_labels = grades[valid_indices]
                    
                    logits = _extract_class_logits(model(batch_images))
                    probs = F.softmax(logits, dim=-1)
                    preds = logits.argmax(dim=-1)
                    
                    for j, (pred, prob, label) in enumerate(zip(preds, probs, batch_labels)):
                        joint_data.append({
                            'pred': pred.item(),
                            'prob': prob.cpu().numpy()
                        })
                        joint_labels.append(label.item())
                        joint_probs.append(prob.cpu().numpy())
        
        if len(joint_labels) > 0:
            preds_list = [d['pred'] for d in joint_data]
            acc = accuracy_score(joint_labels, preds_list)
            results[physical_joint] = acc
            
            print(f"  {physical_joint:12s}: 准确率={acc:.2%}, 样本数={len(joint_labels)}")
            
            probs_array = np.array(joint_probs)
            labels_array = np.array(joint_labels)
            
            if num_classes == 2:
                scores = probs_array[:, 1]
                auc_score = plot_roc_curve(
                    labels_array, scores, 
                    f'{physical_joint} ROC',
                    f'{save_dir}/roc_{physical_joint.lower()}.png'
                )
                auc_results[physical_joint] = auc_score
            else:
                avg_auc, class_aucs = plot_multiclass_roc(
                    labels_array, probs_array,
                    [f'Grade_{i}' for i in range(num_classes)],
                    f'{save_dir}/roc_{physical_joint.lower()}.png',
                    f'{physical_joint} ROC曲线'
                )
                auc_results[physical_joint] = avg_auc
    
    if results:
        overall_avg_acc = np.mean(list(results.values()))
        print(f"\n平均准确率: {overall_avg_acc:.2%}")
    else:
        print("\n警告: 没有有效的评估结果")
        overall_avg_acc = 0.0
    
    if auc_results:
        overall_avg_auc = np.mean(list(auc_results.values()))
        print(f"平均AUC: {overall_avg_auc:.3f}")
        
        plt.figure(figsize=(12, 8))
        names = list(auc_results.keys())
        aucs = list(auc_results.values())
        colors = plt.cm.viridis(np.linspace(0, 1, len(names)))
        
        bars = plt.bar(range(len(names)), aucs, color=colors)
        plt.xticks(range(len(names)), names, rotation=45, ha='right')
        plt.ylabel('AUC Score', fontsize=12)
        plt.xlabel('Joint Category', fontsize=12)
        plt.title('Joint Classification AUC Comparison', fontsize=14)
        plt.ylim([0.5, 1.0])
        plt.axhline(y=0.5, color='r', linestyle='--', label='Random Classifier')
        
        for bar, auc_val in zip(bars, aucs):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{auc_val:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(f'{save_dir}/joint_classification_auc_comparison.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"\n✓ AUC对比图已保存: {save_dir}/joint_classification_auc_comparison.png")
    
    return results


def evaluate_domain_loss(
    trained_models: Dict[str, str],
    source_test: DataLoader,
    target_test: DataLoader,
    device: torch.device,
    save_dir: str = 'plots'
) -> Dict:
    """评估域损失（支持DANN分立模型）
    
    Args:
        trained_models: 训练好的DANN模型路径字典
        source_test: 源域测试数据加载器
        target_test: 目标域测试数据加载器
        device: 设备
        save_dir: 保存目录
    Returns:
        域评估结果字典
    """
    os.makedirs(save_dir, exist_ok=True)
    
    print("\n" + "=" * 60)
    print("              域损失测试")
    print("=" * 60)
    
    all_source_feats = []
    all_target_feats = []
    
    for physical_joint in Config.PHYSICAL_JOINT_NAMES:
        if physical_joint not in trained_models:
            continue
        
        model_path = trained_models[physical_joint]
        if not os.path.exists(model_path):
            continue
        
        try:
            num_classes = get_num_classes_from_model(model_path, device)
            model = load_single_joint_model(model_path, num_classes, device)
            
            with torch.no_grad():
                for batch in source_test:
                    images = batch['image'].to(device)
                    feats = _extract_backbone_features(model, images)
                    all_source_feats.append(feats.cpu())
                
                for batch in target_test:
                    images = batch['image'].to(device)
                    feats = _extract_backbone_features(model, images)
                    all_target_feats.append(feats.cpu())
            
            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
        except Exception as e:
            print(f"  警告: 处理 {physical_joint} 时出错: {e}")
            continue
    
    if len(all_source_feats) == 0 or len(all_target_feats) == 0:
        print("  警告: 没有足够的特征数据")
        return {'mmd': 0.0, 'domain_auc': 0.5, 'domain_acc': 0.5}
    
    source_feats = torch.cat(all_source_feats, dim=0).numpy()
    target_feats = torch.cat(all_target_feats, dim=0).numpy()
    
    def gaussian_kernel(x, y, sigma=1.0):
        diff = x[:, np.newaxis, :] - y[np.newaxis, :, :]
        return np.exp(-np.sum(diff ** 2, axis=-1) / (2 * sigma ** 2))
    
    K_ss = gaussian_kernel(source_feats, source_feats).mean()
    K_tt = gaussian_kernel(target_feats, target_feats).mean()
    K_st = gaussian_kernel(source_feats, target_feats).mean()
    mmd = max(0, K_ss + K_tt - 2 * K_st)
    
    print(f"  MMD域差异: {mmd:.4f} (越小越好，<0.1表示域对齐良好)")
    
    plt.figure(figsize=(10, 8))
    
    plt.subplot(2, 2, 1)
    if len(source_feats) > 0 and len(target_feats) > 0:
        from sklearn.decomposition import PCA
        all_feats = np.vstack([source_feats, target_feats])
        if all_feats.shape[1] > 2:
            pca = PCA(n_components=2)
            all_feats_2d = pca.fit_transform(all_feats)
            source_2d = all_feats_2d[:len(source_feats)]
            target_2d = all_feats_2d[len(source_feats):]
        else:
            source_2d = source_feats[:, :2]
            target_2d = target_feats[:, :2]
        
        plt.scatter(source_2d[:, 0], source_2d[:, 1], c='blue', alpha=0.5, label='源域', s=20)
        plt.scatter(target_2d[:, 0], target_2d[:, 1], c='red', alpha=0.5, label='目标域', s=20)
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.title('域特征分布 (PCA)')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.hist(source_feats[:, 0], bins=30, alpha=0.6, label='源域', color='blue', density=True)
    plt.hist(target_feats[:, 0], bins=30, alpha=0.6, label='目标域', color='red', density=True)
    plt.xlabel('特征值')
    plt.ylabel('密度')
    plt.title('特征分布对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 3)
    plt.text(0.5, 0.6, f'MMD = {mmd:.4f}', fontsize=14, ha='center', transform=plt.gca().transAxes)
    plt.text(0.5, 0.4, 'MMD < 0.1 表示域对齐良好' if mmd < 0.1 else 'MMD >= 0.1 需要更多域适应', 
             fontsize=12, ha='center', transform=plt.gca().transAxes)
    plt.axis('off')
    plt.title('域自适应评估指标')
    
    plt.tight_layout()
    plt.savefig(f'{save_dir}/domain_adaptation_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n✓ 域自适应分析图已保存: {save_dir}/domain_adaptation_analysis.png")
    
    domain_acc = 0.5
    domain_auc = 0.5
    
    return {'mmd': mmd, 'domain_auc': domain_auc, 'domain_acc': domain_acc}


def evaluate_bone_age_mae(
    trained_models: Dict[str, str],
    test_loader: DataLoader,
    device: torch.device,
    save_dir: str = 'plots',
    yolo_model_path: Optional[str] = None,
) -> Dict:
    """评估骨龄预测MAE（支持多关节融合评分）
    
    Args:
        trained_models: 训练好的模型路径字典
        test_loader: RSNA测试数据加载器
        device: 设备
        save_dir: 保存目录
        yolo_model_path: YOLO关节检测权重路径，可选
    Returns:
        MAE评估结果字典
    """
    os.makedirs(save_dir, exist_ok=True)
    
    print("\n" + "=" * 60)
    print("              RSNA骨龄MAE测试")
    print("=" * 60)

    try:
        compute_medical_grades_fn = compute_medical_grades_from_physical_logits
        compute_bone_age_fn = compute_bone_age_months_from_grades
    except NameError:
        try:
            from .bone_age_utils import (
                compute_bone_age_months_from_grades as compute_bone_age_fn,
                compute_medical_grades_from_physical_logits as compute_medical_grades_fn,
            )
        except ImportError:
            from bone_age_utils import (
                compute_bone_age_months_from_grades as compute_bone_age_fn,
                compute_medical_grades_from_physical_logits as compute_medical_grades_fn,
            )

    supports_use_argmax = _supports_kwarg(compute_medical_grades_fn, 'use_argmax')
    supports_interpolate = _supports_kwarg(compute_bone_age_fn, 'interpolate')
    
    loaded_models = {}
    for physical_joint in Config.PHYSICAL_JOINT_NAMES:
        if physical_joint not in trained_models:
            continue
        
        model_path = trained_models[physical_joint]
        if not os.path.exists(model_path):
            continue
        
        num_classes = get_num_classes_from_model(model_path, device)
        model = load_single_joint_model(model_path, num_classes, device)
        loaded_models[physical_joint] = model
    
    if not loaded_models:
        print("  错误: 没有可用的模型")
        return {'mae': 0.0, 'male_mae': 0.0, 'female_mae': 0.0}
    
    print(f"  已加载 {len(loaded_models)} 个关节模型")

    joint_transform = _get_joint_eval_transform(test_loader)
    try:
        segmentor_cls = DPV3JointSegmentor
    except NameError:
        try:
            from .dpv3_segmentation import DPV3JointSegmentor as segmentor_cls
        except ImportError:
            from dpv3_segmentation import DPV3JointSegmentor as segmentor_cls

    try:
        segmentor = segmentor_cls(target_count=23)
    except Exception as e:
        print(f"  ⚠️ DPV3分割器初始化失败，回退到整图推理: {e}")
        segmentor = None

    yolo_model = _load_yolo_joint_model(yolo_model_path)
    if yolo_model is None:
        print("  当前分割模式: DPV3启发式分割（无YOLO检测先验）")
    else:
        print("  当前分割模式: YOLO检测 + DPV3增强分割")

    segmented_samples = 0
    fallback_samples = 0
    debug_log_limit = 5
    debug_log_count = 0
    
    all_preds = []
    all_true = []
    all_genders = []
    dataset = getattr(test_loader, 'dataset', None)
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="RSNA骨龄测试"):
            images = batch['image'].to(device)
            true_ages = batch['bone_age'].cpu().numpy()
            gender_flags = batch['gender'].to(device=device, dtype=torch.float32)
            genders = gender_flags.cpu().numpy()
            image_paths = batch.get('image_path', [None] * images.size(0))

            for idx in range(images.size(0)):
                image_tensor = images[idx:idx + 1]
                raw_image_path = image_paths[idx] if isinstance(image_paths, (list, tuple)) else image_paths
                image_path = _resolve_image_path(raw_image_path, dataset=dataset)
                sample_gender = gender_flags[idx:idx + 1]

                physical_logits = {}
                used_segmentation = False
                segment_debug = {
                    'image_path': raw_image_path,
                    'resolved_image_path': image_path,
                    'image_load_ok': False,
                    'segment_success': False,
                    'segment_error': '',
                    'total_regions': 0,
                    'yolo_count': 0,
                    'bfs_count': 0,
                    'named_region_count': 0,
                    'matched_model_joints': [],
                    'hand_side': 'unknown',
                }

                if segmentor is not None and image_path and os.path.exists(image_path):
                    image_bgr = _load_image_bgr(image_path)
                    if image_bgr is not None:
                        segment_debug['image_load_ok'] = True
                        segment_result = segmentor.segment(image_bgr, yolo_model=yolo_model)
                        segment_debug['segment_success'] = bool(segment_result.get('success', False))
                        segment_debug['segment_error'] = str(segment_result.get('error', '') or '')
                        segment_debug['total_regions'] = int(segment_result.get('total_regions', 0) or 0)
                        segment_debug['yolo_count'] = int(segment_result.get('yolo_count', 0) or 0)
                        segment_debug['bfs_count'] = int(segment_result.get('bfs_count', 0) or 0)
                        segment_debug['hand_side'] = segment_result.get('hand_side', 'unknown')
                        if segment_result.get('success', False) and segment_result.get('regions'):
                            named_regions = _infer_project_named_regions(
                                segment_result.get('regions', []),
                                segment_result.get('hand_side', 'unknown'),
                            )
                            segment_debug['named_region_count'] = len(named_regions)
                            grouped_crop_tensors: Dict[str, List[torch.Tensor]] = {}
                            for region in named_regions:
                                model_joint = region.get('model_joint')
                                if model_joint not in loaded_models:
                                    continue
                                crop_bgr = _crop_region_from_image(image_bgr, region['bbox_coords'])
                                if crop_bgr is None:
                                    continue
                                grouped_crop_tensors.setdefault(model_joint, []).append(
                                    _prepare_crop_tensor(crop_bgr, joint_transform, device)
                                )

                            for physical_joint, crop_tensors in grouped_crop_tensors.items():
                                if not crop_tensors:
                                    continue
                                crop_batch = torch.cat(crop_tensors, dim=0)
                                logits = _extract_class_logits(loaded_models[physical_joint](crop_batch))
                                physical_logits[physical_joint] = logits.mean(dim=0, keepdim=True)

                            segment_debug['matched_model_joints'] = sorted(grouped_crop_tensors.keys())
                            used_segmentation = len(physical_logits) > 0

                for physical_joint, model in loaded_models.items():
                    if physical_joint in physical_logits:
                        continue
                    physical_logits[physical_joint] = _extract_class_logits(model(image_tensor))

                if used_segmentation:
                    segmented_samples += 1
                else:
                    fallback_samples += 1
                    if debug_log_count < debug_log_limit:
                        debug_log_count += 1
                        print(
                            "  [SegDebug] fallback:"
                            f" raw_path={raw_image_path or 'None'},"
                            f" resolved_path={segment_debug['resolved_image_path'] or 'None'},"
                            f" path={os.path.basename(image_path) if image_path else 'None'},"
                            f" image_load_ok={segment_debug['image_load_ok']},"
                            f" success={segment_debug['segment_success']},"
                            f" total_regions={segment_debug['total_regions']},"
                            f" yolo_count={segment_debug['yolo_count']},"
                            f" bfs_count={segment_debug['bfs_count']},"
                            f" named_regions={segment_debug['named_region_count']},"
                            f" matched_joints={segment_debug['matched_model_joints']},"
                            f" hand_side={segment_debug['hand_side']},"
                            f" error={segment_debug['segment_error'] or 'None'}"
                        )

                if supports_use_argmax:
                    predicted_grades = compute_medical_grades_fn(
                        physical_logits,
                        device,
                        use_argmax=True,
                    )
                else:
                    predicted_grades = compute_medical_grades_fn(physical_logits, device)

                if supports_interpolate:
                    pred_age = compute_bone_age_fn(
                        predicted_grades,
                        sample_gender,
                        interpolate=False,
                    )
                else:
                    pred_age = compute_bone_age_fn(predicted_grades, sample_gender)

                all_preds.append(float(pred_age.item()))
                all_true.append(float(true_ages[idx]))
                all_genders.append(float(genders[idx]))
    
    all_preds = np.array(all_preds)
    all_true = np.array(all_true)
    all_genders = np.array(all_genders)

    print(f"  DPV3分割成功样本数: {segmented_samples}")
    print(f"  整图回退样本数: {fallback_samples}")
    
    mae = mean_absolute_error(all_true, all_preds)
    
    male_mask = all_genders > 0.5
    female_mask = ~male_mask
    
    male_mae = mean_absolute_error(all_true[male_mask], all_preds[male_mask]) if male_mask.sum() > 0 else 0
    female_mae = mean_absolute_error(all_true[female_mask], all_preds[female_mask]) if female_mask.sum() > 0 else 0
    
    print(f"  总体MAE: {mae:.2f} 个月")
    print(f"  男童MAE: {male_mae:.2f} 个月")
    print(f"  女童MAE: {female_mae:.2f} 个月")
    
    print("\n骨龄预测可视化分析:")
    
    plt.figure(figsize=(16, 12))
    
    plt.subplot(2, 3, 1)
    errors = all_preds - all_true
    plt.hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='零误差')
    plt.axvline(x=np.mean(errors), color='g', linestyle='-', linewidth=2, label=f'平均误差: {np.mean(errors):.2f}')
    plt.xlabel('预测误差 (月)', fontsize=10)
    plt.ylabel('频数', fontsize=10)
    plt.title('预测误差分布', fontsize=12)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 2)
    plt.scatter(all_true, all_preds, alpha=0.5, s=10, c='steelblue')
    min_val = min(all_true.min(), all_preds.min())
    max_val = max(all_true.max(), all_preds.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='理想预测线')
    
    plt.xlabel('实际骨龄 (月)', fontsize=10)
    plt.ylabel('预测骨龄 (月)', fontsize=10)
    plt.title(f'预测 vs 实际 (MAE={mae:.2f}月)', fontsize=12)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 3)
    male_errors = all_preds[male_mask] - all_true[male_mask]
    female_errors = all_preds[female_mask] - all_true[female_mask]
    
    if len(male_errors) > 0:
        plt.hist(male_errors, bins=30, alpha=0.6, label=f'男童 (MAE={male_mae:.2f})', color='blue', density=True)
    if len(female_errors) > 0:
        plt.hist(female_errors, bins=30, alpha=0.6, label=f'女童 (MAE={female_mae:.2f})', color='red', density=True)
    plt.axvline(x=0, color='black', linestyle='--', linewidth=2)
    plt.xlabel('预测误差 (月)', fontsize=10)
    plt.ylabel('密度', fontsize=10)
    plt.title('男女童误差分布对比', fontsize=12)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 4)
    if all_true.max() > all_true.min():
        age_bins = np.linspace(all_true.min(), all_true.max(), 10)
        bin_mae = []
        bin_centers = []
        for i in range(len(age_bins) - 1):
            mask = (all_true >= age_bins[i]) & (all_true < age_bins[i + 1])
            if mask.sum() > 0:
                bin_mae.append(mean_absolute_error(all_true[mask], all_preds[mask]))
                bin_centers.append((age_bins[i] + age_bins[i + 1]) / 2)
        
        if len(bin_mae) > 0:
            plt.bar(range(len(bin_mae)), bin_mae, color='coral', alpha=0.7, edgecolor='black')
            plt.xticks(range(len(bin_mae)), [f'{int(c)}' for c in bin_centers], rotation=45)
            plt.xlabel('骨龄 (月)', fontsize=10)
            plt.ylabel('MAE (月)', fontsize=10)
            plt.title('不同年龄段MAE', fontsize=12)
            plt.grid(True, alpha=0.3, axis='y')
    
    plt.subplot(2, 3, 5)
    male_true = all_true[male_mask]
    male_preds = all_preds[male_mask]
    female_true = all_true[female_mask]
    female_preds = all_preds[female_mask]
    
    if len(male_true) > 0:
        plt.scatter(male_true, male_preds, alpha=0.5, s=10, c='blue', label='男童')
    if len(female_true) > 0:
        plt.scatter(female_true, female_preds, alpha=0.5, s=10, c='red', label='女童')
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2)
    plt.xlabel('实际骨龄 (月)', fontsize=10)
    plt.ylabel('预测骨龄 (月)', fontsize=10)
    plt.title('男女童预测对比', fontsize=12)
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 3, 6)
    metrics = ['Overall MAE', 'Male MAE', 'Female MAE']
    values = [mae, male_mae, female_mae]
    colors = ['green' if v < 12 else 'orange' if v < 18 else 'red' for v in values]
    bars = plt.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black')
    plt.axhline(y=12, color='orange', linestyle='--', linewidth=1.5, label='可接受阈值 (12月)')
    plt.axhline(y=18, color='red', linestyle='--', linewidth=1.5, label='需改进阈值 (18月)')
    plt.ylabel('MAE (个月)', fontsize=10)
    plt.title('MAE性能指标', fontsize=12)
    plt.legend(fontsize=8, loc='upper right')
    
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{save_dir}/bone_age_prediction_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ 骨龄预测分析图已保存: {save_dir}/bone_age_prediction_analysis.png")
    
    plt.figure(figsize=(8, 6))
    plt.scatter(all_true, all_preds, alpha=0.5, s=15, c='steelblue')
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='理想预测线')
    plt.xlabel('实际骨龄 (月)', fontsize=12)
    plt.ylabel('预测骨龄 (月)', fontsize=12)
    plt.title(f'骨龄预测 (MAE={mae:.2f}月)', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/bone_age_prediction_scatter.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ 骨龄预测散点图已保存: {save_dir}/bone_age_prediction_scatter.png")
    
    return {
        'mae': mae,
        'male_mae': male_mae,
        'female_mae': female_mae
    }


def plot_roc_curves(results_dir: str = './results'):
    """绘制ROC曲线"""
    print("\n绘制ROC曲线...")
    os.makedirs(results_dir, exist_ok=True)
    print(f"  (ROC曲线绘制需要在完整测试后生成)")


## 🔄 Stage2: 域自适应泛化训练

使用GRL（梯度反转层）和MMD（最大均值差异）进行域自适应

In [15]:
"""
Stage2 Kaggle模型微调启动脚本

使用方法:
1. 修改下面的配置（KAGGLE_MODEL_DIR, TARGET_DATA_DIR等）
2. 运行: python stage2_kaggle_finetune.py
"""

import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
import torch.nn as nn
import pandas as pd
import sys
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm



#os.chdir(r'c:\D\codeWorkplace\pythonworkplace\pythonworkplace\gulingyuce\gulingyuce\cleann\Medical-Bone-Age\训练代码\1')

KAGGLE_MODEL_DIR = r"/kaggle/input/models/lihaohao111/jointrecognizemodels2-0/pytorch/default/1/小关节分级最新版模型"

SOURCE_DATA_DIR = Config.SOURCE_DATA_ROOT
TARGET_DATA_DIR = Config.TARGET_DATA_ROOT

MODEL_NAME_MAPPING = {
    'Radius': 'stage1_Radius_model.pth',
    'Ulna': 'stage1_Ulna_model.pth',
    'MCPFirst': 'stage1_MCPFirst_model.pth',
    'MCP': 'stage1_MCPFirst_model.pth',
    'PIPFirst': 'stage1_PIPFirst_model.pth',
    'PIP': 'stage1_PIPFirst_model.pth',
    'DIPFirst': 'stage1_DIPFirst_model.pth',
    'DIP': 'stage1_DIPFirst_model.pth',
    'MIPFirst': 'stage1_MIP_model.pth',
    'MIP': 'stage1_MIP_model.pth',
}


def choose_safe_batch_size(device: torch.device) -> int:
    """为 Kaggle DANN 训练选择更稳妥的 batch size。"""
    if device.type != 'cuda':
        return min(Config.BATCH_SIZE, 8)

    try:
        total_gb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    except Exception:
        total_gb = 0

    if total_gb >= 15:
        return min(Config.BATCH_SIZE, 8)
    if total_gb >= 10:
        return min(Config.BATCH_SIZE, 4)
    return min(Config.BATCH_SIZE, 2)


def build_retry_batch_sizes(initial_batch_size: int):
    """构建OOM时的自动降批次重试序列。"""
    retry_sizes = []
    current = max(1, int(initial_batch_size))
    while current >= 1:
        if current not in retry_sizes:
            retry_sizes.append(current)
        if current == 1:
            break
        current = max(1, current // 2)
    return retry_sizes

def prepare_kaggle_model_dict():
    """准备Kaggle模型路径字典"""
    kaggle_models = {}

    for code_joint, model_filename in MODEL_NAME_MAPPING.items():
        model_path = os.path.join(KAGGLE_MODEL_DIR, model_filename)
        if os.path.exists(model_path):
            try:
                checkpoint = torch.load(model_path, weights_only=True)
                keys = list(checkpoint.keys())
                
                has_fc = 'fc.weight' in checkpoint or 'fc.0.weight' in checkpoint or 'fc.1.weight' in checkpoint or 'classifier.1.weight' in checkpoint
                
                fc_keys = [k for k in keys if 'fc' in k.lower() or 'classifier' in k.lower()]
                
                print(f"  ✅ {code_joint} -> {model_filename}")
                print(f"     权重键数量: {len(keys)}")
                
                if fc_keys:
                    print(f"     包含分类层! 键: {fc_keys}")
                    for fk in fc_keys:
                        if 'weight' in fk:
                            shape = checkpoint[fk].shape
                            print(f"       {fk}: {shape} → 类别数={shape[0]}")
                else:
                    print(f"     仅backbone权重")
                
                if len(keys) <= 10:
                    print(f"     键列表: {keys}")
                else:
                    print(f"     前10个键: {keys[:10]}")
                
            except Exception as e:
                print(f"  ⚠️ {code_joint} -> {model_filename} (加载失败: {e})")
                continue
            
            kaggle_models[code_joint] = model_path
        else:
            print(f"  ⚠️ {code_joint} -> {model_filename} (文件不存在)")

    return kaggle_models

def prepare_target_data():
    """准备目标域数据 (使用目录结构，与main.py一致)
    
    支持两种目录结构:
    1. 无等级: 关节/图片.png
    2. 有等级: 关节/等级/图片.png
    
    注意: 目标域数据不需要分级标签，训练中使用默认等级
    """
    print("\n准备目标域数据...")

    if not os.path.exists(TARGET_DATA_DIR):
        print(f"❌ 目标数据目录不存在: {TARGET_DATA_DIR}")
        print("\n请准备目标域数据目录，支持两种结构：")
        print("\n结构1 (无等级，推荐用于目标域):")
        print("  TARGET_DATA_DIR/")
        print("  ├── Radius/")
        print("  │   ├── image1.png")
        print("  │   └── image2.png")
        print("  ├── Ulna/")
        print("  │   ├── image3.png")
        print("  │   └── image4.png")
        print("  └── ...")
        print("\n结构2 (有等级):")
        print("  TARGET_DATA_DIR/")
        print("  ├── Radius/")
        print("  │   ├── 1/")
        print("  │   │   └── image1.png")
        print("  │   └── 2/")
        print("  │       └── image2.png")
        print("  └── ...")
        return None, None, None, None

    print(f"\n使用目录结构加载目标域数据: {TARGET_DATA_DIR}")
    
    target_train, target_val, target_test = split_arthrosis_dataset(
        TARGET_DATA_DIR,
        train_ratio=0.667,
        val_ratio=0.333,
        test_ratio=0.333,
        default_grade=1
    )

    print(f"  ✅ 目标训练数据: {len(target_train)} 条")
    print(f"  ✅ 目标验证数据: {len(target_val)} 条")
    print(f"  ✅ 目标测试数据: {len(target_test)} 条")

    img_size = Config.IMG_SIZE
    target_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    return target_train, target_val, target_test, target_transform


def prepare_source_data():
    """准备源域数据，用于真正的DANN训练。"""
    print("\n准备源域数据...")

    if not os.path.exists(SOURCE_DATA_DIR):
        print(f"  ⚠️ 未找到源域数据目录: {SOURCE_DATA_DIR}")
        print("  将回退到目标域监督微调或直接保留Stage1模型")
        return pd.DataFrame()

    print(f"  使用目录结构加载源域数据: {SOURCE_DATA_DIR}")
    source_train, source_val, source_test = split_arthrosis_dataset(
        SOURCE_DATA_DIR,
        train_ratio=Config.TRAIN_RATIO,
        val_ratio=Config.VAL_RATIO,
        test_ratio=Config.TEST_RATIO,
    )

    print(f"  ✅ 源训练数据: {len(source_train)} 条")
    print(f"  ✅ 源验证数据: {len(source_val)} 条")
    print(f"  ✅ 源测试数据: {len(source_test)} 条")
    return source_train

def run_stage2_finetune():
    """运行Stage2 Kaggle模型微调"""
    print("="*70)
    print("Stage2 Kaggle模型微调")
    print("="*70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n使用设备: {device}")

    print("\n步骤1: 检查Kaggle模型...")
    kaggle_models = prepare_kaggle_model_dict()

    if not kaggle_models:
        print("❌ 没有可用的Kaggle模型！")
        return

    print(f"\n  可用模型数量: {len(kaggle_models)}")

    print("\n步骤2: 准备目标域数据...")
    result = prepare_target_data()

    if result[0] is None:
        return

    target_train, target_val, target_test, target_transform = result

    print("\n步骤3: 准备源域数据...")
    source_train = prepare_source_data()

    print("\n步骤4: 配置Stage2训练参数...")
    safe_batch_size = choose_safe_batch_size(device)
    training_config = {
        'num_epochs': Config.EPOCHS_STAGE2,
        'batch_size': safe_batch_size,
        'learning_rate': Config.LEARNING_RATE_STAGE2,
        'weight_decay': 1e-4,
        'early_stop_patience': 5,
        'use_dann': True,
        'domain_adversarial_weight': 0.3,
        'training_mode': 'balanced_dann',
        'max_source_target_ratio': 8,
        'min_source_samples': 32,
    }

    print(f"  训练轮数: {training_config['num_epochs']}")
    print(f"  批次大小: {training_config['batch_size']}")
    print(f"  学习率: {training_config['learning_rate']}")
    print(f"  DANN对抗权重: {training_config['domain_adversarial_weight']}")
    print(f"  训练模式: {training_config['training_mode']}")
    if training_config['training_mode'] == 'balanced_dann':
        print(f"  源域采样倍率上限: {training_config['max_source_target_ratio']}x")
        print(f"  每关节最小保留源域样本: {training_config['min_source_samples']}")

    print("\n步骤5: 开始Stage2微调...")
    print("  这将使用Kaggle预训练模型在目标域上进行域自适应训练")
    if training_config['training_mode'] == 'balanced_dann':
        print("  策略: 平衡DANN域自适应 + Kaggle预训练权重")
    else:
        print("  策略: DANN域自适应 + Kaggle预训练权重")
    print(f"  OOM自动重试批次序列: {build_retry_batch_sizes(training_config['batch_size'])}")

    try:
        stage2_models = None
        last_error = None
        for attempt_batch_size in build_retry_batch_sizes(training_config['batch_size']):
            try:
                print(f"\n  尝试使用 batch_size={attempt_batch_size} 进行Stage2训练...")
                clear_gpu_memory()
                stage2_models = stage2_domain_adaptation(
                    stage1_models=kaggle_models,
                    target_train=target_train,
                    device=device,
                    num_epochs=training_config['num_epochs'],
                    batch_size=attempt_batch_size,
                    train_transform=target_transform,
                    source_train=source_train,
                    training_mode=training_config['training_mode'],
                    max_source_target_ratio=training_config['max_source_target_ratio'],
                    min_source_samples=training_config['min_source_samples'],
                )
                break
            except torch.OutOfMemoryError as oom_error:
                last_error = oom_error
                clear_gpu_memory()
                print(f"  ⚠️ batch_size={attempt_batch_size} 显存不足，准备降批次重试")
                if attempt_batch_size == 1:
                    raise
            except RuntimeError as runtime_error:
                if 'out of memory' not in str(runtime_error).lower():
                    raise
                last_error = runtime_error
                clear_gpu_memory()
                print(f"  ⚠️ batch_size={attempt_batch_size} 触发OOM，准备降批次重试")
                if attempt_batch_size == 1:
                    raise

        if stage2_models is None and last_error is not None:
            raise last_error

        print("\n" + "="*70)
        print("Stage2微调完成！")
        print("="*70)
        print(f"训练了 {len(stage2_models)} 个关节模型")
        print("模型保存在当前目录")

    except Exception as e:
        print(f"\n❌ Stage2微调失败: {e}")
        import traceback
        traceback.print_exc()
        print("\n常见问题：")
        print("1. 目标域数据格式是否正确？")
        print("2. Kaggle模型路径是否正确？")
        print("3. 关节名称是否匹配？")
        print("4. 类别数是否匹配？")

def main():
    print("Stage2 Kaggle模型微调工具")
    print("="*70)

    print("\n请确保:")
    print("1. Kaggle模型已下载到指定目录")
    print("2. 目标域数据目录已准备好（结构：关节/图片 或 关节/等级/图片）")
    print("3. 关节名称与代码中一致")

    response = 'y'
    if response.lower() == 'y':
        run_stage2_finetune()
    else:
        print("已取消")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n已中断训练")
    except Exception as e:
        print(f"\n❌ 发生错误: {e}")
        import traceback
        traceback.print_exc()


Stage2 Kaggle模型微调工具

请确保:
1. Kaggle模型已下载到指定目录
2. 目标域数据目录已准备好（结构：关节/图片 或 关节/等级/图片）
3. 关节名称与代码中一致
Stage2 Kaggle模型微调

使用设备: cuda

步骤1: 检查Kaggle模型...
  ✅ Radius -> stage1_Radius_model.pth
     权重键数量: 320
     包含分类层! 键: ['fc.1.weight', 'fc.1.bias']
       fc.1.weight: torch.Size([14, 2048]) → 类别数=14
     前10个键: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean']
  ✅ Ulna -> stage1_Ulna_model.pth
     权重键数量: 320
     包含分类层! 键: ['fc.1.weight', 'fc.1.bias']
       fc.1.weight: torch.Size([12, 2048]) → 类别数=12
     前10个键: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean']
  ✅ MCPFirst -> stage1_MCPFirst_model.pth
     权重键数量: 320
     包含分类层! 键: ['fc.1.weight', 'fc.1.bias']
       fc.1.weight

  Epoch 1/30 [α=0.000]: 100%|██████████| 4/4 [00:03<00:00,  1.29it/s, cls=2.4194, dom=0.7280, mmd=0.6250, obj=2.6385]


    Epoch 1/30 [LR=1.00e-05]: 分类=3.2981, 域=0.7431, MMD=0.6250, 总损失=3.5217


  Epoch 2/30 [α=0.165]: 100%|██████████| 4/4 [00:00<00:00,  4.47it/s, cls=3.1818, dom=0.6369, mmd=0.6250, obj=3.3735]


    Epoch 2/30 [LR=1.00e-05]: 分类=3.1306, 域=0.6997, MMD=0.6250, 总损失=3.3411


...

  Epoch 30/30 [α=1.000]: 100%|██████████| 7/7 [00:01<00:00,  3.56it/s, cls=1.4888, dom=0.6395, mmd=0.2679, obj=1.6809]

  Epoch 30/30 [α=1.000]: 100%|██████████| 7/7 [00:01<00:00,  3.56it/s, cls=1.4888, dom=0.6395, mmd=0.2679, obj=1.6809]

    Epoch 30/30 [LR=1.00e-05]: 分类=1.7510, 域=0.6943, MMD=0.2679, 总损失=1.9596
  ✓ 最佳域损失模型已保存: stage2_MIP_best_domain_loss_model.pth


  ✓ Stage2模型已保存: stage2_MIP_model.pth (best_loss=1.9596)


  ✓ MIP显存已释放

Stage2 DANN域自适应训练完成！共 10 个模型

Stage2微调完成！
训练了 10 个关节模型
模型保存在当前目录


In [16]:
"""
骨龄公式与关节映射工具

统一训练与评估中的:
- RUS-CHN 评分转骨龄
- 年/月单位换算
- 物理关节 -> 医学关节映射
- 评分表插值
"""

import math
from typing import Dict, List

import torch
import torch.nn.functional as F




YEARS_TO_MONTHS = 12.0

RUS_CHN_MALE_COEFFS = [
    2.01790023656577,
    -0.0931820870747269,
    0.00334709095418796,
    -3.32988302362153e-05,
    1.75712910819776e-07,
    -5.59998691223273e-10,
    1.1296711294933e-12,
    -1.45218037113138e-15,
    1.15333377080353e-18,
    -5.15887481551927e-22,
    9.94098428102335e-26,
]

RUS_CHN_FEMALE_COEFFS = [
    5.81191794824917,
    -0.271546561737745,
    0.00526301486340724,
    -4.37797717401925e-05,
    2.0858722025667e-07,
    -6.21879866563429e-10,
    1.19909931745368e-12,
    -1.49462900826936e-15,
    1.162435538672e-18,
    -5.12713017846218e-22,
    9.78989966891478e-26,
]


def _build_medical_to_physical_mapping() -> Dict[str, List[str]]:
    mapping = {medical_joint: [] for medical_joint in Config.MEDICAL_JOINT_NAMES}

    for physical_joint, medical_joints in Config.PHYSICAL_TO_MEDICAL_MAPPING.items():
        for medical_joint in medical_joints:
            mapping.setdefault(medical_joint, [])
            if physical_joint not in mapping[medical_joint]:
                mapping[medical_joint].append(physical_joint)

    for medical_joint in Config.MEDICAL_JOINT_NAMES:
        if not mapping[medical_joint]:
            mapping[medical_joint] = [medical_joint]

    return mapping


MEDICAL_TO_PHYSICAL_MAPPING = _build_medical_to_physical_mapping()


def calc_bone_age_years_from_score(score: float, gender: str) -> float:
    """根据 RUS-CHN 总分计算骨龄，返回单位: 年。"""
    coeffs = RUS_CHN_MALE_COEFFS if gender == 'male' else RUS_CHN_FEMALE_COEFFS

    bone_age = coeffs[0]
    for power, coeff in enumerate(coeffs[1:], start=1):
        bone_age += coeff * math.pow(score, power)

    return max(0.0, bone_age)


def calc_bone_age_months_from_score(score: float, gender: str) -> float:
    """根据 RUS-CHN 总分计算骨龄，返回单位: 月。"""
    return calc_bone_age_years_from_score(score, gender) * YEARS_TO_MONTHS


def calc_bone_age_years_from_score_tensor(scores: torch.Tensor, genders: torch.Tensor) -> torch.Tensor:
    """根据 RUS-CHN 总分计算骨龄，返回单位: 年。"""
    coeff_male = torch.as_tensor(RUS_CHN_MALE_COEFFS, device=scores.device, dtype=scores.dtype)
    coeff_female = torch.as_tensor(RUS_CHN_FEMALE_COEFFS, device=scores.device, dtype=scores.dtype)

    score_terms = [torch.ones_like(scores)]
    current_power = torch.ones_like(scores)
    for _ in range(1, len(coeff_male)):
        current_power = current_power * scores
        score_terms.append(current_power)

    score_poly = torch.stack(score_terms, dim=1)

    male_bone_age = (score_poly * coeff_male.unsqueeze(0)).sum(dim=1)
    female_bone_age = (score_poly * coeff_female.unsqueeze(0)).sum(dim=1)

    bone_age = genders * male_bone_age + (1 - genders) * female_bone_age
    return torch.clamp(bone_age, min=0)


def calc_bone_age_months_from_score_tensor(scores: torch.Tensor, genders: torch.Tensor) -> torch.Tensor:
    """根据 RUS-CHN 总分计算骨龄，返回单位: 月。"""
    return calc_bone_age_years_from_score_tensor(scores, genders) * YEARS_TO_MONTHS


def interpolate_score_tensor(grades: torch.Tensor, score_table: List[float]) -> torch.Tensor:
    """对浮点等级做线性插值，输出对应评分。"""
    table = torch.as_tensor(score_table, device=grades.device, dtype=grades.dtype)
    max_idx = table.size(0) - 1

    clamped_grades = torch.clamp(grades, 0, float(max_idx))
    lower = torch.floor(clamped_grades).long()
    upper = torch.clamp(lower + 1, max=max_idx)
    frac = clamped_grades - lower.to(grades.dtype)

    return table[lower] + frac * (table[upper] - table[lower])


def lookup_score_tensor(grades: torch.Tensor, score_table: List[float]) -> torch.Tensor:
    """对离散等级直接查表取分。"""
    table = torch.as_tensor(score_table, device=grades.device, dtype=grades.dtype)
    max_idx = table.size(0) - 1
    discrete_grades = torch.clamp(grades.long(), 0, max_idx)
    return table[discrete_grades]


def compute_medical_grades_from_physical_logits(
    physical_logits: Dict[str, torch.Tensor],
    device: torch.device,
    use_argmax: bool = False,
) -> torch.Tensor:
    """将物理关节 logits 转成医学关节的期望等级。"""
    if not physical_logits:
        raise ValueError("physical_logits 为空，无法计算医学关节等级")

    sample_logits = next(iter(physical_logits.values()))
    batch_size = sample_logits.size(0)
    dtype = sample_logits.dtype

    medical_grade_predictions = []
    for medical_joint in Config.MEDICAL_JOINT_NAMES:
        physical_joints = MEDICAL_TO_PHYSICAL_MAPPING.get(medical_joint, [medical_joint])

        joint_logits_list = [physical_logits[pj] for pj in physical_joints if pj in physical_logits]

        if joint_logits_list:
            max_classes = max(logits.size(1) for logits in joint_logits_list)
            aligned_logits = []
            for logits in joint_logits_list:
                if logits.size(1) < max_classes:
                    pad = torch.full(
                        (batch_size, max_classes - logits.size(1)),
                        float('-inf'),
                        device=logits.device,
                        dtype=logits.dtype,
                    )
                    logits = torch.cat([logits, pad], dim=1)
                aligned_logits.append(logits)

            fused_logits = torch.stack(aligned_logits).mean(dim=0)
            probs = F.softmax(fused_logits, dim=1)
            if use_argmax:
                expected_grade = probs.argmax(dim=1).to(dtype)
            else:
                grade_indices = torch.arange(probs.size(1), device=device, dtype=probs.dtype).unsqueeze(0)
                expected_grade = (probs * grade_indices).sum(dim=1)
        else:
            default_num_classes = Config.MEDICAL_JOINT_NUM_CLASSES.get(medical_joint, 1)
            default_grade = float(max(default_num_classes - 1, 0)) / 2.0
            expected_grade = torch.full((batch_size,), default_grade, device=device, dtype=dtype)

        medical_grade_predictions.append(expected_grade)

    return torch.stack(medical_grade_predictions, dim=1)


def compute_bone_age_months_from_grades(
    predicted_grades: torch.Tensor,
    gender_flags: torch.Tensor,
    interpolate: bool = True,
) -> torch.Tensor:
    """从医学关节等级计算骨龄，返回单位: 月。"""
    batch_size = predicted_grades.size(0)

    male_scores = torch.zeros(batch_size, device=predicted_grades.device, dtype=predicted_grades.dtype)
    female_scores = torch.zeros(batch_size, device=predicted_grades.device, dtype=predicted_grades.dtype)

    for joint_index, medical_joint in enumerate(Config.MEDICAL_JOINT_NAMES):
        if medical_joint in Config.SCORE_TABLE['male']:
            if interpolate:
                male_scores += interpolate_score_tensor(
                    predicted_grades[:, joint_index],
                    Config.SCORE_TABLE['male'][medical_joint],
                )
            else:
                male_scores += lookup_score_tensor(
                    predicted_grades[:, joint_index],
                    Config.SCORE_TABLE['male'][medical_joint],
                )

        if medical_joint in Config.SCORE_TABLE['female']:
            if interpolate:
                female_scores += interpolate_score_tensor(
                    predicted_grades[:, joint_index],
                    Config.SCORE_TABLE['female'][medical_joint],
                )
            else:
                female_scores += lookup_score_tensor(
                    predicted_grades[:, joint_index],
                    Config.SCORE_TABLE['female'][medical_joint],
                )

    total_scores = gender_flags * male_scores + (1 - gender_flags) * female_scores
    return calc_bone_age_months_from_score_tensor(total_scores, gender_flags)


In [17]:

"""
DPV3 关节分割模块

包含:
- BFSAutoClustering: BFS自动聚类分块
- UnionFindMerge: Union-Find去重合并
- DPGrayExpansion: DP灰度扩展算法
- DPV3JointSegmentor: DPV3关节分割器
- DPV3Visualizer: DPV3分割可视化器
"""

import cv2
import numpy as np
from typing import List, Dict, Tuple
from collections import deque


class BFSAutoClustering:
    """BFS自动聚类分块 - 从灰度图像中提取连通区域"""
    
    def __init__(self, gray_tolerance: int = 10, min_area: int = 100):
        self.gray_tolerance = gray_tolerance
        self.min_area = min_area

    def cluster(self, gray_image: np.ndarray, mask: np.ndarray = None) -> List[Dict]:
        if mask is None:
            mask = np.zeros_like(gray_image)

        h, w = gray_image.shape
        visited = np.zeros((h, w), dtype=bool)
        blocks = []

        def bfs_flood_fill(start_x: int, start_y: int) -> Tuple[List, float, float]:
            pixels = []
            queue = deque([(start_x, start_y)])
            visited[start_y, start_x] = True
            base_gray = gray_image[start_y, start_x]

            while queue:
                x, y = queue.popleft()
                pixels.append((x, y))

                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = x + dx, y + dy

                    if 0 <= nx < w and 0 <= ny < h:
                        if not visited[ny, nx] and mask[ny, nx] == 0:
                            pixel_gray = gray_image[ny, nx]
                            if abs(int(pixel_gray) - int(base_gray)) <= self.gray_tolerance:
                                visited[ny, nx] = True
                                queue.append((nx, ny))

            if pixels:
                pixel_values = [gray_image[y, x] for x, y in pixels]
                gray_mean = np.mean(pixel_values)
                gray_std = np.std(pixel_values)
            else:
                gray_mean = base_gray
                gray_std = 0

            return pixels, gray_mean, gray_std

        for y in range(h):
            for x in range(w):
                if not visited[y, x] and mask[y, x] == 0:
                    pixels, gray_mean, gray_std = bfs_flood_fill(x, y)

                    if len(pixels) > self.min_area:
                        xs, ys = zip(*pixels)
                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)
                        cx = (x_min + x_max) // 2
                        cy = (y_min + y_max) // 2

                        blocks.append({
                            "pixels": pixels,
                            "centroid": (cx, cy),
                            "area": len(pixels),
                            "bbox": (x_min, y_min, x_max - x_min, y_max - y_min),
                            "bbox_coords": (x_min, y_min, x_max, y_max),
                            "gray_mean": gray_mean,
                            "gray_std": gray_std,
                            "gray_min": min([gray_image[y, x] for x, y in pixels]) if pixels else 0,
                            "gray_max": max([gray_image[y, x] for x, y in pixels]) if pixels else 255,
                        })

        blocks.sort(key=lambda b: b["area"], reverse=True)
        return blocks


class UnionFindMerge:
    """Union-Find去重合并重叠的分块"""
    
    def __init__(self, overlap_threshold: float = 0.3):
        self.overlap_threshold = overlap_threshold

    def merge(self, blocks: List[Dict]) -> List[Dict]:
        if not blocks:
            return []

        n = len(blocks)
        parent = list(range(n))
        rank = [0] * n

        def find(x: int) -> int:
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x: int, y: int):
            px, py = find(x), find(y)
            if px == py:
                return
            if rank[px] < rank[py]:
                px, py = py, px
            parent[py] = px
            if rank[px] == rank[py]:
                rank[px] += 1

        def boxes_overlap(box1: Tuple, box2: Tuple) -> float:
            x1_min, y1_min, x1_max, y1_max = box1
            x2_min, y2_min, x2_max, y2_max = box2

            x_overlap = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
            y_overlap = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
            overlap_area = x_overlap * y_overlap

            if overlap_area == 0:
                return 0.0

            area1 = (x1_max - x1_min) * (y1_max - y1_min)
            area2 = (x2_max - x2_min) * (y2_max - y2_min)
            min_area = min(area1, area2)

            return overlap_area / min_area if min_area > 0 else 0.0

        for i in range(n):
            for j in range(i + 1, n):
                overlap_ratio = boxes_overlap(blocks[i]["bbox_coords"], blocks[j]["bbox_coords"])
                if overlap_ratio > self.overlap_threshold:
                    union(i, j)

        merged_groups = {}
        for i in range(n):
            root = find(i)
            if root not in merged_groups:
                merged_groups[root] = []
            merged_groups[root].append(blocks[i])

        merged_blocks = []
        for root, group in merged_groups.items():
            if len(group) == 1:
                merged_blocks.append(group[0])
            else:
                all_pixels = []
                all_x, all_y = [], []
                gray_means = []

                for block in group:
                    all_pixels.extend(block["pixels"])
                    all_x.extend([p[0] for p in block["pixels"]])
                    all_y.extend([p[1] for p in block["pixels"]])
                    gray_means.append(block["gray_mean"])

                x_min, x_max = min(all_x), max(all_x)
                y_min, y_max = min(all_y), max(all_y)
                cx = (x_min + x_max) // 2
                cy = (y_min + y_max) // 2

                merged_blocks.append({
                    "pixels": all_pixels,
                    "centroid": (cx, cy),
                    "area": len(all_pixels),
                    "bbox": (x_min, y_min, x_max - x_min, y_max - y_min),
                    "bbox_coords": (x_min, y_min, x_max, y_max),
                    "gray_mean": np.mean(gray_means),
                    "gray_std": np.std(gray_means) if len(gray_means) > 1 else 0,
                    "gray_min": min([p[0] for p in all_pixels]) if all_pixels else 0,
                    "gray_max": max([p[0] for p in all_pixels]) if all_pixels else 255,
                })

        merged_blocks.sort(key=lambda b: b["area"], reverse=True)
        return merged_blocks


class DPGrayExpansion:
    """DP灰度扩展算法 - 找到恰好target_count个骨骼的最佳灰度范围"""
    
    def __init__(self, max_expansion: int = 80, step: int = 2):
        self.max_expansion = max_expansion
        self.step = step

    def expand(self, gray_image: np.ndarray, merged_blocks: List[Dict],
               yolo_count: int, initial_range: Tuple[int, int],
               target_count: int) -> Tuple[Tuple[int, int], List]:
        
        init_low, init_high = initial_range
        target_bfs_count = target_count - yolo_count

        def count_bfs_bones(gray_low: int, gray_high: int) -> int:
            count = 0
            for block in merged_blocks:
                block_gray_min = block["gray_min"]
                block_gray_max = block["gray_max"]

                if block_gray_max >= gray_low and block_gray_min <= gray_high:
                    area = block["area"]
                    if area > 0:
                        count += 1
            return count

        dp_history = []
        initial_bfs_count = count_bfs_bones(init_low, init_high)
        total_count = yolo_count + initial_bfs_count
        dp_history.append((init_low, init_high, initial_bfs_count, total_count))

        low, high = init_low, init_high

        for i in range(self.max_expansion // self.step):
            new_low = max(0, low - self.step)
            new_high = min(255, high + self.step)

            bfs_count = count_bfs_bones(new_low, new_high)
            total = yolo_count + bfs_count
            dp_history.append((new_low, new_high, bfs_count, total))

            if total >= target_count:
                return (new_low, new_high), dp_history

            low, high = new_low, new_high

        best_range = init_low, init_high
        best_total = total_count
        min_diff = abs(total_count - target_count)

        for l in range(max(0, init_low - self.max_expansion), min(256, init_high + self.max_expansion), self.step):
            for h in range(l, min(256, init_high + self.max_expansion), self.step):
                bfs_count = count_bfs_bones(l, h)
                total = yolo_count + bfs_count
                diff = abs(total - target_count)

                if diff < min_diff:
                    min_diff = diff
                    best_total = total
                    best_range = (l, h)

        return best_range, dp_history


class DPV3JointSegmentor:
    """
    DPV3关节分割器
    
    整合了:
    - BFS聚类分块
    - Union-Find去重合并
    - DP灰度扩展
    - 中心扩展法
    """
    
    def __init__(self, target_count: int = 23):
        self.target_count = target_count
        self.yolo_imgsz = 1024
        
    def segment(self, image: np.ndarray, yolo_model=None, conf: float = 0.5) -> Dict:
        """
        执行DPV3关节分割
        
        Args:
            image: 输入图像 (RGB或BGR)
            yolo_model: YOLO模型 (可选)
            conf: YOLO置信度阈值
            
        Returns:
            分割结果字典
        """
        try:
            orig_h, orig_w = image.shape[:2]
            
            if len(image.shape) == 3:
                gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            else:
                gray = image.copy()
            
            yolo_regions = []
            if yolo_model is not None:
                results = yolo_model.predict(
                    image,
                    imgsz=self.yolo_imgsz,
                    conf=conf,
                    verbose=False,
                )
                if results and len(results) > 0:
                    for box in results[0].boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        cls_id = int(box.cls[0])
                        cls_name = results[0].names[cls_id]
                        yolo_regions.append({
                            "label": cls_name,
                            "centroid": ((int(x1) + int(x2)) // 2, (int(y1) + int(y2)) // 2),
                            "bbox_coords": (int(x1), int(y1), int(x2), int(y2)),
                        })
            
            detected_regions = []
            if len(yolo_regions) < self.target_count:
                yolo_mask = np.zeros_like(gray)
                for region in yolo_regions:
                    x1, y1, x2, y2 = region["bbox_coords"]
                    padding = 5
                    x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
                    x2, y2 = min(orig_w, x2 + padding), min(orig_h, y2 + padding)
                    yolo_mask[y1:y2, x1:x2] = 255
                
                bfs_clustering = BFSAutoClustering()
                initial_blocks = bfs_clustering.cluster(gray, yolo_mask)
                
                union_find = UnionFindMerge()
                merged_blocks = union_find.merge(initial_blocks)
                
                if yolo_regions:
                    bone_pixels = []
                    for region in yolo_regions:
                        x1, y1, x2, y2 = region["bbox_coords"]
                        x1, y1 = max(0, x1), max(0, y1)
                        x2, y2 = min(orig_w, x2), min(orig_h, y2)
                        roi = gray[y1:y2, x1:x2]
                        if roi.size > 0:
                            _, bone_mask = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                            bone_pixels_in_roi = roi[bone_mask > 0]
                            if len(bone_pixels_in_roi) > 0:
                                bone_pixels.extend(bone_pixels_in_roi.tolist())
                    
                    if bone_pixels:
                        bone_mean = np.mean(bone_pixels)
                        bone_std = np.std(bone_pixels)
                        init_low = max(0, int(bone_mean - 1.5 * bone_std))
                        init_high = min(255, int(bone_mean + 1.5 * bone_std))
                    else:
                        init_low, init_high = 80, 180
                else:
                    init_low, init_high = 80, 180
                
                dp_expansion = DPGrayExpansion()
                best_gray_range, dp_history = dp_expansion.expand(
                    gray, merged_blocks, len(yolo_regions), (init_low, init_high), self.target_count
                )
                
                low, high = best_gray_range
                detected_regions = []
                for block in merged_blocks:
                    block_gray_min = block["gray_min"]
                    block_gray_max = block["gray_max"]
                    
                    if block_gray_max >= low and block_gray_min <= high:
                        area = block["area"]
                        if 500 <= area <= 20000:
                            detected_regions.append({
                                "label": "BFS_Bone",
                                "centroid": block["centroid"],
                                "bbox_coords": block["bbox_coords"],
                                "area": block["area"],
                                "source": "bfs_dp"
                            })
            
            all_regions = yolo_regions + detected_regions
            
            radius_regions = [r for r in all_regions if r.get("label") == "Radius"]
            ulna_regions = [r for r in all_regions if r.get("label") == "Ulna"]
            if radius_regions and ulna_regions:
                radius_cx = radius_regions[0]["centroid"][0]
                ulna_cx = ulna_regions[0]["centroid"][0]
                hand_side = "left" if ulna_cx < radius_cx else "right"
            else:
                hand_side = "unknown"
            
            return {
                "success": True,
                "hand_side": hand_side,
                "total_regions": len(all_regions),
                "yolo_count": len(yolo_regions),
                "bfs_count": len(detected_regions),
                "best_gray_range": best_gray_range,
                "regions": all_regions,
                "merged_blocks": len(merged_blocks),
            }
            
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "regions": []
            }
    
    def crop_joint_regions(self, image: np.ndarray, regions: List[Dict], 
                         joint_size: int = 224) -> Dict[str, np.ndarray]:
        """裁剪各个关节区域"""
        cropped_joints = {}
        
        for region in regions:
            label = region.get("label", "Unknown")
            x1, y1, x2, y2 = region["bbox_coords"]
            
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)
            
            roi = image[y1:y2, x1:x2]
            
            if roi.size > 0:
                roi_resized = cv2.resize(roi, (joint_size, joint_size))
                cropped_joints[label] = roi_resized
        
        return cropped_joints


class DPV3Visualizer:
    """DPV3分割可视化器"""
    
    def __init__(self, target_count: int = 23):
        self.target_count = target_count
        
    def segment_and_visualize(self, image_path: str, yolo_model=None, save_path: str = None):
        """
        分割并可视化结果
        
        Args:
            image_path: 输入图像路径
            yolo_model: YOLO模型 (可选)
            save_path: 保存路径 (可选)
        """
        import matplotlib.pyplot as plt
        import os
        
        image = cv2.imread(image_path)
        if image is None:
            print(f"无法读取图像: {image_path}")
            return
        
        orig_h, orig_w = image.shape[:2]
        
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()
        
        yolo_regions = []
        if yolo_model is not None:
            results = yolo_model.predict(image, conf=0.5, verbose=False)
            if results and len(results) > 0:
                for box in results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    cls_id = int(box.cls[0])
                    cls_name = results[0].names[cls_id]
                    yolo_regions.append({
                        "label": cls_name,
                        "centroid": (int((x1 + x2) // 2), int((y1 + y2) // 2)),
                        "bbox_coords": (int(x1), int(y1), int(x2), int(y2)),
                    })
        
        yolo_mask = np.zeros_like(gray)
        for region in yolo_regions:
            x1, y1, x2, y2 = region["bbox_coords"]
            padding = 5
            x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
            x2, y2 = min(orig_w, x2 + padding), min(orig_h, y2 + padding)
            yolo_mask[y1:y2, x1:x2] = 255
        
        bfs_clustering = BFSAutoClustering()
        initial_blocks = bfs_clustering.cluster(gray, yolo_mask)
        
        union_find = UnionFindMerge()
        merged_blocks = union_find.merge(initial_blocks)
        
        if yolo_regions:
            bone_pixels = []
            for region in yolo_regions:
                x1, y1, x2, y2 = region["bbox_coords"]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(orig_w, x2), min(orig_h, y2)
                roi = gray[y1:y2, x1:x2]
                if roi.size > 0:
                    _, bone_mask = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                    bone_pixels_in_roi = roi[bone_mask > 0]
                    if len(bone_pixels_in_roi) > 0:
                        bone_pixels.extend(bone_pixels_in_roi.tolist())
            
            if bone_pixels:
                bone_mean = np.mean(bone_pixels)
                bone_std = np.std(bone_pixels)
                init_low = max(0, int(bone_mean - 1.5 * bone_std))
                init_high = min(255, int(bone_mean + 1.5 * bone_std))
            else:
                init_low, init_high = 80, 180
        else:
            init_low, init_high = 80, 180
        
        dp_expansion = DPGrayExpansion()
        best_gray_range, dp_history = dp_expansion.expand(
            gray, merged_blocks, len(yolo_regions), (init_low, init_high), self.target_count
        )
        
        low, high = best_gray_range
        detected_regions = []
        for block in merged_blocks:
            block_gray_min = block["gray_min"]
            block_gray_max = block["gray_max"]
            
            if block_gray_max >= low and block_gray_min <= high:
                detected_regions.append({
                    "label": "BFS_Bone",
                    "centroid": block["centroid"],
                    "bbox_coords": block["bbox_coords"],
                    "area": block["area"],
                    "source": "bfs_dp"
                })
        
        all_regions = yolo_regions + detected_regions
        
        hand_side = "unknown"
        radius_regions = [r for r in all_regions if r.get("label") == "Radius"]
        ulna_regions = [r for r in all_regions if r.get("label") == "Ulna"]
        if radius_regions and ulna_regions:
            radius_cx = radius_regions[0]["centroid"][0]
            ulna_cx = ulna_regions[0]["centroid"][0]
            hand_side = "LEFT" if ulna_cx < radius_cx else "RIGHT"
        
        self._visualize_results(image, all_regions, hand_side, best_gray_range, 
                              len(yolo_regions), len(detected_regions), save_path)
        
        return all_regions
    
    def _visualize_results(self, image: np.ndarray, regions: List[Dict], 
                          hand_side: str, gray_range: Tuple[int, int],
                          yolo_count: int, bfs_count: int, save_path: str = None):
        """绘制可视化结果"""
        import matplotlib.pyplot as plt
        
        vis = image.copy()
        
        yolo_color = (0, 255, 0)
        for region in regions:
            if region.get("source") != "bfs_dp":
                x1, y1, x2, y2 = region["bbox_coords"]
                cv2.rectangle(vis, (x1, y1), (x2, y2), yolo_color, 2)
                cx, cy = region["centroid"]
                cv2.circle(vis, (cx, cy), 5, yolo_color, -1)
                label = region.get("label", "Unknown")
                cv2.putText(vis, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, yolo_color, 1)
        
        bfs_color = (255, 0, 0)
        for region in regions:
            if region.get("source") == "bfs_dp":
                x1, y1, x2, y2 = region["bbox_coords"]
                cv2.rectangle(vis, (x1, y1), (x2, y2), bfs_color, 2)
                cx, cy = region["centroid"]
                cv2.circle(vis, (cx, cy), 5, bfs_color, -1)
        
        panel_y = 25
        cv2.putText(vis, f"Hand: {hand_side}", (10, panel_y), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(vis, f"YOLO: {yolo_count}", (10, panel_y + 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, yolo_color, 2)
        cv2.putText(vis, f"BFS: {bfs_count}", (10, panel_y + 55), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, bfs_color, 2)
        cv2.putText(vis, f"Gray Range: {gray_range}", (10, panel_y + 80), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(vis, f"Total: {len(regions)}", (10, panel_y + 105), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        
        plt.figure(figsize=(15, 10))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.title(f"DPV3 Segmentation Results\nHand: {hand_side} | YOLO: {yolo_count} | BFS: {bfs_count} | Total: {len(regions)}", 
                 fontsize=14)
        plt.axis('off')
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"图像已保存: {save_path}")
        
        plt.show()
    
    def visualize_crops(self, image: np.ndarray, regions: List[Dict], 
                        crop_size: int = 100, max_show: int = 20):
        """展示裁剪后的关节图像"""
        import matplotlib.pyplot as plt
        
        all_crops = []
        for region in regions:
            x1, y1, x2, y2 = region["bbox_coords"]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)
            
            roi = image[y1:y2, x1:x2]
            if roi.size > 0:
                roi_resized = cv2.resize(roi, (crop_size, crop_size))
                label = region.get("label", "Unknown")
                source = region.get("source", "yolo")
                all_crops.append({
                    "image": roi_resized,
                    "label": label,
                    "source": source,
                    "area": region.get("area", 0)
                })
        
        all_crops.sort(key=lambda x: x["area"], reverse=True)
        all_crops = all_crops[:max_show]
        
        n = len(all_crops)
        cols = min(5, n)
        rows = (n + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
        if rows == 1:
            axes = axes.reshape(1, -1)
        if cols == 1:
            axes = axes.reshape(-1, 1)
        
        for idx, crop in enumerate(all_crops):
            row = idx // cols
            col = idx % cols
            ax = axes[row, col]
            
            img_rgb = cv2.cvtColor(crop["image"], cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            ax.set_title(f"{crop['label']}\n({crop['source']})", fontsize=8)
            ax.axis('off')
        
        for idx in range(n, rows * cols):
            row = idx // cols
            col = idx % cols
            axes[row, col].axis('off')
        
        plt.suptitle(f"Cropped Joint Regions (showing {n}/{len(regions)})", fontsize=14)
        plt.tight_layout()
        plt.show()


def test_dpv3_segmentation(image_path: str, yolo_model_path: str = None):
    """
    测试DPV3分割效果
    
    Args:
        image_path: 输入图像路径
        yolo_model_path: YOLO模型路径 (可选)
    """
    print("=" * 60)
    print("         DPV3 关节分割测试")
    print("=" * 60)
    
    yolo_model = None
    if yolo_model_path:
        import os
        if os.path.exists(yolo_model_path):
            try:
                from ultralytics import YOLO
                yolo_model = YOLO(yolo_model_path)
                print(f"YOLO模型已加载: {yolo_model_path}")
            except ImportError:
                print("Warning: ultralytics未安装，将跳过YOLO检测")
    
    visualizer = DPV3Visualizer(target_count=23)
    
    print(f"\n处理图像: {image_path}")
    regions = visualizer.segment_and_visualize(image_path, yolo_model)
    
    if regions:
        print(f"\n分割结果统计:")
        print(f"  - 总区域数: {len(regions)}")
        print(f"  - YOLO检测: {sum(1 for r in regions if r.get('source') != 'bfs_dp')}")
        print(f"  - BFS检测: {sum(1 for r in regions if r.get('source') == 'bfs_dp')}")
        
        image = cv2.imread(image_path)
        if image is not None:
            print("\n展示裁剪后的关节图像...")
            visualizer.visualize_crops(image, regions)
    
    return regions


In [18]:
!pip install ultralytics
"""
模型评估脚本

评估stage1和stage2模型在Arthritis数据集和RSNA数据集上的性能：
1. 加载stage1模型
2. 加载stage2模型（包括最佳域损失模型）
3. 评估Arthritis数据集上的关节分类准确率
4. 计算RSNA数据集上的骨龄MAE

使用方法:
    python evaluate_models.py
"""

import os
import sys
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models




def load_stage1_models(model_dir: str = '.') -> dict:
    """加载stage1模型"""
    stage1_models = {}
    
    for joint in Config.PHYSICAL_JOINT_NAMES:
        model_path = os.path.join(model_dir, f'stage1_{joint}_model.pth')
        if os.path.exists(model_path):
            stage1_models[joint] = model_path
            print(f"  ✓ 加载Stage1模型: {joint}")
        else:
            print(f"  ⚠ 跳过Stage1模型: {joint} (文件不存在)")
    
    return stage1_models


def load_stage2_models(model_dir: str = '.') -> dict:
    """加载stage2模型"""
    stage2_models = {}
    
    for joint in Config.PHYSICAL_JOINT_NAMES:
        model_path = os.path.join(model_dir, f'stage2_{joint}_model.pth')
        if os.path.exists(model_path):
            stage2_models[joint] = model_path
            print(f"  ✓ 加载Stage2模型: {joint}")
        else:
            print(f"  ⚠ 跳过Stage2模型: {joint} (文件不存在)")
    
    return stage2_models


def load_best_domain_loss_models(model_dir: str = '.') -> dict:
    """加载最佳域损失模型"""
    best_models = {}
    
    for joint in Config.PHYSICAL_JOINT_NAMES:
        model_path = os.path.join(model_dir, f'stage2_{joint}_best_domain_loss_model.pth')
        if os.path.exists(model_path):
            best_models[joint] = model_path
            print(f"  ✓ 加载最佳域损失模型: {joint}")
        else:
            print(f"  ⚠ 跳过最佳域损失模型: {joint} (文件不存在)")
    
    return best_models


def prepare_arthritis_data():
    """准备Arthritis数据集（使用main.py的划分比例）"""
    print("\n准备Arthritis数据集...")
    
    if not os.path.exists(Config.SOURCE_DATA_ROOT):
        print(f"  ⚠ 源域数据不存在: {Config.SOURCE_DATA_ROOT}")
        return None, None
    
    train_data, val_data, test_data = split_arthrosis_dataset(
        Config.SOURCE_DATA_ROOT,
        Config.TRAIN_RATIO,
        Config.VAL_RATIO,
        Config.TEST_RATIO
    )
    
    print(f"  使用main.py的配置:")
    print(f"    训练集: {len(train_data)} 条 (TRAIN_RATIO={Config.TRAIN_RATIO})")
    print(f"    验证集: {len(val_data)} 条 (VAL_RATIO={Config.VAL_RATIO})")
    print(f"    测试集: {len(test_data)} 条 (TEST_RATIO={Config.TEST_RATIO})")
    
    return train_data, test_data


def prepare_rsna_data():
    """准备RSNA数据集（使用main.py的配置）"""
    print("\n准备RSNA数据集...")
    
    if not os.path.exists(Config.RSNA_CSV_PATH):
        print(f"  ⚠ RSNA CSV不存在: {Config.RSNA_CSV_PATH}")
        return None
    elif not os.path.exists(Config.RSNA_IMG_DIR):
        print(f"  ⚠ RSNA图像目录不存在: {Config.RSNA_IMG_DIR}")
        return None
    
    train_data, val_data, test_data = prepare_rsna_dataset(
        Config.RSNA_CSV_PATH,
        Config.RSNA_IMG_DIR,
        Config.RSNA_SAMPLES
    )
    
    print(f"  使用main.py的配置:")
    print(f"    训练集: {len(train_data)} 条 (RSNA_SAMPLES={Config.RSNA_SAMPLES})")
    print(f"    验证集: {len(val_data)} 条")
    print(f"    测试集: {len(test_data)} 条")
    
    return test_data


def evaluate_models():
    """执行完整的模型评估流程"""
    print("=" * 70)
    print("          Stage1 & Stage2 模型评估")
    print("=" * 70)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n使用设备: {device}")
    
    print("\n步骤1: 加载模型...")
    
    stage1_models = load_stage1_models(r'/kaggle/input/models/lihaohao111/jointrecognizemodels2-0/pytorch/default/1/小关节分级最新版模型')
    print(f"\nStage1模型总数: {len(stage1_models)}")
    
    stage2_models = load_stage2_models('/kaggle/working/')
    print(f"\nStage2模型总数: {len(stage2_models)}")
    
    best_domain_models = load_best_domain_loss_models('.')
    print(f"\n最佳域损失模型总数: {len(best_domain_models)}")
    
    if not stage1_models and not stage2_models and not best_domain_models:
        print("\n❌ 没有找到任何模型！")
        print("\n请确保以下文件存在:")
        print("  - stage1_*.pth (Stage1模型)")
        print("  - stage2_*.pth (Stage2模型)")
        print("  - stage2_*_best_domain_loss_model.pth (最佳域损失模型)")
        return
    
    print("\n步骤2: 准备数据集...")
    
    arthritis_data = prepare_arthritis_data()
    rsna_data = prepare_rsna_data()
    
    if arthritis_data is None and rsna_data is None:
        print("\n❌ 没有可用的数据集！")
        return
    
    test_transform = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    results = {}
    
    if arthritis_data is not None:
        arthritis_train, arthritis_test = arthritis_data
        arthritis_dataset = ArthrosisDataset(arthritis_test, transform=test_transform)
        arthritis_loader = DataLoader(
            arthritis_dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=False
        )
        
        print(f"\nArthritis测试样本数: {len(arthritis_test)}")
        
        print("\n" + "=" * 70)
        print("          Stage1 模型评估 (Arthritis数据集)")
        print("=" * 70)
        
        if stage1_models:
             pass
        else:
            print("  ⚠ Stage1模型为空，跳过评估")
        
        print("\n" + "=" * 70)
        print("          Stage2 模型评估 (Arthritis数据集)")
        print("=" * 70)
        
        # if stage2_models:
        #     stage2_results = evaluate_joint_classification(
        #         stage2_models,
        #         arthritis_loader,
        #         device,
        #         save_dir='plots/evaluation/stage2'
        #     )
        #     results['stage2_arthritis'] = stage2_results
            
        #     if stage2_results:
        #         avg_acc = np.mean(list(stage2_results.values()))
        #         print(f"\n✓ Stage2平均准确率: {avg_acc:.2%}")
        # else:
        #     print("  ⚠ Stage2模型为空，跳过评估")
        
        print("\n" + "=" * 70)
        print("          最佳域损失模型评估 (Arthritis数据集)")
        print("=" * 70)
        
        # if best_domain_models:
        #     best_results = evaluate_joint_classification(
        #         best_domain_models,
        #         arthritis_loader,
        #         device,
        #         save_dir='plots/evaluation/best_domain_loss'
        #     )
        #     results['best_domain_loss_arthritis'] = best_results
            
        #     if best_results:
        #         avg_acc = np.mean(list(best_results.values()))
        #         print(f"\n✓ 最佳域损失模型平均准确率: {avg_acc:.2%}")
        # else:
        #     print("  ⚠ 最佳域损失模型为空，跳过评估")
    
    if rsna_data is not None and len(rsna_data) > 0:
        rsna_dataset = RSNADataset(rsna_data, Config.RSNA_IMG_DIR, transform=test_transform)
        rsna_loader = DataLoader(
            rsna_dataset, 
            batch_size=16, 
            shuffle=False
        )
        
        print(f"\nRSNA测试样本数: {len(rsna_data)}")
        
        print("\n" + "=" * 70)
        print("          Stage1 模型 RSNA MAE评估")
        print("=" * 70)
        
        if stage1_models:
            stage1_mae_results = evaluate_bone_age_mae(
                stage1_models,
                rsna_loader,
                device,
                save_dir='plots/evaluation/rsna_stage1',
                    yolo_model_path='/kaggle/input/models/lihaohao111/bestyolo/pytorch/default/1/best (1).pt'
            )
            results['stage1_rsna_mae'] = stage1_mae_results
            
            if stage1_mae_results:
                print(f"\n✓ Stage1 RSNA MAE: {stage1_mae_results.get('mae', 0):.2f} 个月")
                print(f"  男童MAE: {stage1_mae_results.get('male_mae', 0):.2f} 个月")
                print(f"  女童MAE: {stage1_mae_results.get('female_mae', 0):.2f} 个月")
        else:
            print("  ⚠ Stage1模型为空，跳过评估")
        
        print("\n" + "=" * 70)
        print("          Stage2 模型 RSNA MAE评估")
        print("=" * 70)
        
        if stage2_models:
            stage2_mae_results = evaluate_bone_age_mae(
                stage2_models,
                rsna_loader,
                device,
                save_dir='plots/evaluation/rsna_stage2',
                    yolo_model_path='/kaggle/input/models/lihaohao111/bestyolo/pytorch/default/1/best (1).pt'
            )
            results['stage2_rsna_mae'] = stage2_mae_results
            
            if stage2_mae_results:
                print(f"\n✓ Stage2 RSNA MAE: {stage2_mae_results.get('mae', 0):.2f} 个月")
                print(f"  男童MAE: {stage2_mae_results.get('male_mae', 0):.2f} 个月")
                print(f"  女童MAE: {stage2_mae_results.get('female_mae', 0):.2f} 个月")
        else:
            print("  ⚠ Stage2模型为空，跳过评估")
        
        print("\n" + "=" * 70)
        print("          最佳域损失模型 RSNA MAE评估")
        print("=" * 70)
        
        if best_domain_models:
            best_mae_results = evaluate_bone_age_mae(
                best_domain_models,
                rsna_loader,
                device,
                save_dir='plots/evaluation/rsna_best_domain_loss',
                    yolo_model_path='/kaggle/input/models/lihaohao111/bestyolo/pytorch/default/1/best (1).pt'
            )
            results['best_domain_loss_rsna_mae'] = best_mae_results
            
            if best_mae_results:
                print(f"\n✓ 最佳域损失模型 RSNA MAE: {best_mae_results.get('mae', 0):.2f} 个月")
                print(f"  男童MAE: {best_mae_results.get('male_mae', 0):.2f} 个月")
                print(f"  女童MAE: {best_mae_results.get('female_mae', 0):.2f} 个月")
        else:
            print("  ⚠ 最佳域损失模型为空，跳过评估")
    
    print("\n" + "=" * 70)
    print("          评估结果总结")
    print("=" * 70)
    
    #if 'stage1_arthritis' in results and results['stage1_arthritis']:
        #stage1_acc = np.mean(list(results['stage1_arthritis'].values()))
        #print(f"Stage1 Arthritis准确率: {stage1_acc:.2%}")
    
    # if 'stage2_arthritis' in results and results['stage2_arthritis']:
    #     stage2_acc = np.mean(list(results['stage2_arthritis'].values()))
    #     print(f"Stage2 Arthritis准确率: {stage2_acc:.2%}")
    
    # if 'best_domain_loss_arthritis' in results and results['best_domain_loss_arthritis']:
    #     best_acc = np.mean(list(results['best_domain_loss_arthritis'].values()))
    #     print(f"最佳域损失模型 Arthritis准确率: {best_acc:.2%}")
    
    if 'stage1_rsna_mae' in results and results['stage1_rsna_mae']:
        print(f"Stage1 RSNA MAE: {results['stage1_rsna_mae'].get('mae', 0):.2f} 个月")
    
    # if 'stage2_rsna_mae' in results and results['stage2_rsna_mae']:
    #     print(f"Stage2 RSNA MAE: {results['stage2_rsna_mae'].get('mae', 0):.2f} 个月")
    
    if 'best_domain_loss_rsna_mae' in results and results['best_domain_loss_rsna_mae']:
        print(f"最佳域损失模型 RSNA MAE: {results['best_domain_loss_rsna_mae'].get('mae', 0):.2f} 个月")
    
    print("\n评估完成！")
    print("结果图保存目录:")
    print("  - plots/evaluation/stage1/")
    print("  - plots/evaluation/stage2/")
    print("  - plots/evaluation/best_domain_loss/")
    print("  - plots/evaluation/rsna_stage1/")
    print("  - plots/evaluation/rsna_stage2/")
    print("  - plots/evaluation/rsna_best_domain_loss/")
    
    save_summary(results)
    
    return results


def save_summary(results: dict):
    """保存评估结果摘要"""
    summary_file = 'evaluation_summary.txt'
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("=" * 70 + "\n")
        f.write("          模型评估结果摘要\n")
        f.write("=" * 70 + "\n\n")
        
        f.write("1. Arthritis数据集分类准确率:\n")
        if 'stage1_arthritis' in results and results['stage1_arthritis']:
            f.write("   Stage1模型:\n")
            for joint, acc in results['stage1_arthritis'].items():
                f.write(f"     {joint:12s}: {acc:.2%}\n")
            stage1_avg = np.mean(list(results['stage1_arthritis'].values()))
            f.write(f"     {'平均':12s}: {stage1_avg:.2%}\n\n")
        
        if 'stage2_arthritis' in results and results['stage2_arthritis']:
            f.write("   Stage2模型:\n")
            for joint, acc in results['stage2_arthritis'].items():
                f.write(f"     {joint:12s}: {acc:.2%}\n")
            stage2_avg = np.mean(list(results['stage2_arthritis'].values()))
            f.write(f"     {'平均':12s}: {stage2_avg:.2%}\n\n")
        
        if 'best_domain_loss_arthritis' in results and results['best_domain_loss_arthritis']:
            f.write("   最佳域损失模型:\n")
            for joint, acc in results['best_domain_loss_arthritis'].items():
                f.write(f"     {joint:12s}: {acc:.2%}\n")
            best_avg = np.mean(list(results['best_domain_loss_arthritis'].values()))
            f.write(f"     {'平均':12s}: {best_avg:.2%}\n\n")
        
        f.write("\n2. RSNA数据集骨龄MAE:\n")
        if 'stage1_rsna_mae' in results and results['stage1_rsna_mae']:
            mae = results['stage1_rsna_mae']
            f.write(f"   Stage1模型:\n")
            f.write(f"     总体MAE: {mae.get('mae', 0):.2f} 个月\n")
            f.write(f"     男童MAE: {mae.get('male_mae', 0):.2f} 个月\n")
            f.write(f"     女童MAE: {mae.get('female_mae', 0):.2f} 个月\n\n")
        
        if 'stage2_rsna_mae' in results and results['stage2_rsna_mae']:
            mae = results['stage2_rsna_mae']
            f.write(f"   Stage2模型:\n")
            f.write(f"     总体MAE: {mae.get('mae', 0):.2f} 个月\n")
            f.write(f"     男童MAE: {mae.get('male_mae', 0):.2f} 个月\n")
            f.write(f"     女童MAE: {mae.get('female_mae', 0):.2f} 个月\n\n")
        
        if 'best_domain_loss_rsna_mae' in results and results['best_domain_loss_rsna_mae']:
            mae = results['best_domain_loss_rsna_mae']
            f.write(f"   最佳域损失模型:\n")
            f.write(f"     总体MAE: {mae.get('mae', 0):.2f} 个月\n")
            f.write(f"     男童MAE: {mae.get('male_mae', 0):.2f} 个月\n")
            f.write(f"     女童MAE: {mae.get('female_mae', 0):.2f} 个月\n\n")
        
        f.write("\n" + "=" * 70 + "\n")
    
    print(f"\n✓ 评估结果摘要已保存: {summary_file}")


def main():
    """主函数"""
    print("Stage1 & Stage2 模型评估工具")
    print("=" * 70)
    
    print("\n此脚本将评估以下模型:")
    print("  1. Stage1模型 (stage1_*.pth)")
    print("  2. Stage2模型 (stage2_*.pth)")
    print("  3. 最佳域损失模型 (stage2_*_best_domain_loss_model.pth)")
    
    print("\n评估数据集:")
    print("  1. Arthritis数据集 - 关节分类准确率")
    print("  2. RSNA数据集 - 骨龄MAE")
    
    response = 'y'
    if response.lower() != 'y':
        print("已取消")
        return
    
    evaluate_models()


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n已中断评估")
    except Exception as e:
        print(f"\n❌ 发生错误: {e}")
        import traceback
        traceback.print_exc()


^C
Fatal Python error: init_import_site: Failed to import the site module
Python runtime state: initialized
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1176, in exec_module
  File "<frozen site>", line 652, in <module>
  File "<frozen site>", line 639, in main
  File "<frozen site>", line 421, in addsitepackages
  File "<frozen site>", line 253, in addsitedir
  File "<frozen site>", line 212, in addpackage
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/_numba_cuda_redirector.py", line 2, in <module>
    import importlib.abc
  File "/usr/lib/python3.12/importlib/abc.py", line 18, in <module>
    from .resources import abc as _resources_abc
  File "/usr/lib/python3.12/importlib/resources/__init__.py", 

RSNA骨龄测试:   0%|          | 0/10 [00:00<?, ?it/s]

RSNA骨龄测试:  10%|█         | 1/10 [01:14<11:10, 74.49s/it]

RSNA骨龄测试:  20%|██        | 2/10 [02:35<10:24, 78.09s/it]

RSNA骨龄测试:  30%|███       | 3/10 [04:18<10:28, 89.84s/it]

...

RSNA骨龄测试:  50%|█████     | 5/10 [07:23<07:36, 91.36s/it]

RSNA骨龄测试:  60%|██████    | 6/10 [08:50<05:58, 89.71s/it]

RSNA骨龄测试:  70%|███████   | 7/10 [10:27<04:36, 92.05s/it]

RSNA骨龄测试:  80%|████████  | 8/10 [12:15<03:14, 97.13s/it]

RSNA骨龄测试:  90%|█████████ | 9/10 [13:49<01:36, 96.22s/it]

## 📐 Stage3: 语义对齐 + 骨龄精调

使用RSNA数据集和RUS-CHN公式进行骨龄精调

## 💾 保存模型

## 📈 第六步：评估模型

运行完整的评估流程